# Census Tract Preprocessing Pipeline
## Zero Vehicle Rate for NYC Taxi Zones

This notebook processes the US Census ACS B08141 dataset and produces one **zero vehicle rate** per NYC taxi zone.  
The rate is the proportion of households with no privately owned vehicle.  
A higher rate indicates greater car-free dependency and therefore higher expected taxi demand.  
The rate is then merged into the main hourly taxi demand dataset as a model feature.

> **Feature direction:** `zero_vehicle_rate` is high in transit-dense zones (Manhattan ~72%) and low in car-dependent suburban zones (Staten Island ~5%). Higher rate → more transit-dependent → more taxi demand.

**Input files required**
- `car_ownership.csv` — Census ACS B08141 table (NYC census tracts)
- `census_tract_fips_code.xlsx` — US Census Bureau FIPS reference file for NYC county derivation
- `tl_2024_36_tract.shp` — Census TIGER tract shapefile for New York State
- `taxi_zones.shp` — NYC TLC taxi zone shapefile
- `taxi_zone_lookup.csv` — NYC TLC zone lookup table
- `df_hourly.parquet` — Main taxi demand dataset from TLC preprocessing

**Output produced**
- `zero_vehicle_rate_by_zone.parquet` — Zero vehicle rate per zone (262 zones, exact float precision)
- `zero_vehicle_rate_by_zone.csv` — Human readable backup
- `df_hourly` — Updated with `zero_vehicle_rate` column added


## Imports and File Paths

In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import plotly.express as px
import json
from shapely.ops import unary_union
from scipy import stats as sp_stats
import warnings

warnings.filterwarnings('ignore')
%matplotlib inline

# ── File paths — update to match your local folder structure ──────────────
CSV_PATH    = '../data/census_acs/car_ownership.csv'
TRACT_SHP   = '../data/census_acs/tl_2024_36_tract.shp'
TAXI_SHP    = '../data/taxi_zones/taxi_zones.shp'
LOOKUP_PATH = '../data/taxi_zones/taxi_zone_lookup.csv'
HOURLY_PATH = '../data/processed/taxi_demand_hourly_padded.parquet'
FIPS_PATH   = '../data/census_acs/census_tract_fips_code.xlsx'
OUTPUT_PATH = '../data/processed/'

# ── Borough display constants ─────────────────────────────────────────────
BOROUGH_ORDER  = ['Manhattan', 'Bronx', 'Brooklyn', 'Queens', 'Staten Island']
BOROUGH_COLORS = {
    'Manhattan':    '#534AB7',
    'Bronx':        '#D4537E',
    'Brooklyn':     '#D85A30',
    'Queens':       '#1D9E75',
    'Staten Island':'#BA7517'
}

# ── Seaborn theme ─────────────────────────────────────────────────────────
sns.set_theme(style='whitegrid', font_scale=1.05)
plt.rcParams.update({
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'figure.dpi':        120,
    'savefig.bbox':      'tight'
})

# ── Populated in Step 8 from actual data — never hardcoded ────────────────
BOROUGH_MEANS = {}

print("All imports loaded successfully.")
print(f"BOROUGH_ORDER  = {BOROUGH_ORDER}")
print(f"BOROUGH_COLORS = {list(BOROUGH_COLORS.keys())}")


## NYC County Derivation from FIPS Reference File

NYC county FIPS codes are derived from the official US Census Bureau FIPS reference file.  
Filtering on `CLASSFP = 'H6'` identifies the five NYC consolidated city-county boroughs.  
All 57 other New York State counties have `CLASSFP = 'H1'` (standard county).  
This approach is more robust than hardcoding because it is tied to the official Census classification.

**References**
- US Census Bureau. (2023, May 1). *American National Standards Institute (ANSI), Federal Information Processing Series (FIPS), and other standardized geographic codes.* Census.gov. https://www.census.gov/library/reference/code-lists/ansi.html#cou
- US Census Bureau. (2021, October 8). *Class codes and definitions.* Census.gov. https://www.census.gov/library/reference/code-lists/class-codes.html


In [ ]:
# Load the official US Census Bureau FIPS reference file
fips_df = pd.read_excel(FIPS_PATH)

# CLASSFP = 'H6' identifies NYC's five consolidated city-county boroughs only
# All other NY counties have CLASSFP = 'H1'
nyc_fips = fips_df[fips_df['CLASSFP'] == 'H6'].copy()
nyc_fips['COUNTYFP'] = nyc_fips['COUNTYFP'].astype(str).str.zfill(3)

# NYC_COUNTIES derived from file — never hardcoded
NYC_COUNTIES = nyc_fips['COUNTYFP'].tolist()
assert len(NYC_COUNTIES) == 5, \
    f"Expected 5 NYC counties from CLASSFP='H6', got {len(NYC_COUNTIES)}: {NYC_COUNTIES}"

# Map COUNTYFP to shapefile borough names
# (FIPS COUNTYNAME uses legal names e.g. 'Kings County' → shapefile uses 'Brooklyn')
FIPS_TO_BOROUGH = {
    '005': 'Bronx',
    '047': 'Brooklyn',
    '061': 'Manhattan',
    '081': 'Queens',
    '085': 'Staten Island'
}
nyc_fips['borough_name'] = nyc_fips['COUNTYFP'].map(FIPS_TO_BOROUGH)
assert nyc_fips['borough_name'].notna().all(), \
    f"FIPS_TO_BOROUGH mapping produced NaN for: {nyc_fips[nyc_fips['borough_name'].isna()]['COUNTYFP'].tolist()}"
COUNTY_TO_BOROUGH = dict(zip(nyc_fips['COUNTYFP'], nyc_fips['borough_name']))

print("NYC counties derived from FIPS reference file (CLASSFP = H6):")
print(nyc_fips[['COUNTYFP', 'COUNTYNAME', 'CLASSFP']].to_string(index=False))
print()
print(f"NYC_COUNTIES = {NYC_COUNTIES}")
print(f"COUNTY_TO_BOROUGH = {COUNTY_TO_BOROUGH}")

In [ ]:
# Styled table showing the derived NYC counties — for the methodology chapter
fips_display = nyc_fips[['COUNTYFP', 'COUNTYNAME', 'CLASSFP', 'borough_name']].copy()
fips_display.columns = ['COUNTYFP', 'Legal County Name', 'CLASSFP', 'Shapefile Borough Name']
fips_display = fips_display.reset_index(drop=True)

# Cross-check: confirm derived counties match actual CSV data
df_check      = pd.read_csv(CSV_PATH, skiprows=[1])
csv_counties  = sorted(df_check['GEO_ID'].str.replace('1400000US','',regex=False).str[2:5].unique())
assert sorted(NYC_COUNTIES) == csv_counties, \
    f"FIPS file counties {sorted(NYC_COUNTIES)} do not match CSV counties {csv_counties}"

fips_display['In car_ownership.csv'] = fips_display['COUNTYFP'].isin(csv_counties).map({True: '✓ Yes', False: '✗ No'})

(fips_display.style
 .set_caption('Table: NYC Counties Derived from CLASSFP = H6 (Official Census Classification)')
 .set_properties(**{'text-align': 'left', 'padding': '6px 12px', 'font-size': '13px'})
 .set_table_styles([
     {'selector': 'caption', 'props': [('font-size','14px'),('font-weight','bold'),('padding','8px 0')]},
     {'selector': 'th', 'props': [('background-color','#534AB7'),('color','white'),('font-weight','bold'),('padding','7px 12px')]},
     {'selector': 'tr:nth-child(even)', 'props': [('background-color','#f5f4ff')]},
 ])
 .hide(axis='index'))


---
## Phase 1: Census Data Understanding

### Step 1 — Load census CSV

In [ ]:
# Load the file and skip row 1 which contains label descriptions not data
# Row 0: column codes (B08141_001E etc.)  |  Row 1: human-readable labels → skipped
# With skiprows=[1], columns auto-convert to int64 immediately
df_raw = pd.read_csv(CSV_PATH, skiprows=[1])

print(f"Rows loaded:    {len(df_raw)}")
print(f"Columns loaded: {df_raw.shape[1]}")
print()
print("First 3 rows of key columns:")
print(df_raw[['GEO_ID', 'B08141_001E', 'B08141_002E', 'B08141_002M']].head(3))

In [ ]:
# Show the ghost header problem — why skiprows=[1] is essential
df_with_ghost    = pd.read_csv(CSV_PATH)
df_without_ghost = pd.read_csv(CSV_PATH, skiprows=[1])

ghost_demo = pd.DataFrame({
    'Row':         ['Row 0 (header)',  'Row 1 WITHOUT skiprows=[1]', 'Row 0 WITH skiprows=[1]'],
    'GEO_ID':      [df_raw.columns[0], df_with_ghost.iloc[0,0],      df_raw.iloc[0,0]],
    'B08141_001E': [df_raw.columns[2], df_with_ghost.iloc[0,2],      str(df_raw.iloc[0,2])],
    'dtype':       ['(column name)',   str(df_with_ghost['GEO_ID'].dtype), str(df_raw['GEO_ID'].dtype)]
})

print(f"Raw shape (wrong):   {df_with_ghost.shape}  — all columns load as object")
print(f"Correct shape:       {df_raw.shape}  — numeric columns are int64 immediately")
print()

(ghost_demo.style
 .set_caption('Table: Ghost Header Row Problem — Why skiprows=[1] is Required')
 .set_properties(**{'text-align': 'left', 'padding': '6px 12px', 'font-size': '13px'})
 .set_table_styles([
     {'selector': 'caption', 'props': [('font-size','14px'),('font-weight','bold'),('padding','8px 0')]},
     {'selector': 'th', 'props': [('background-color','#3a3a5c'),('color','white'),('padding','7px 12px')]},
     {'selector': 'tr:nth-child(2)', 'props': [('background-color','#ffeaea'),('color','#a32d2d')]},
     {'selector': 'tr:nth-child(3)', 'props': [('background-color','#e8f5e9'),('color','#085041')]},
 ])
 .hide(axis='index'))


### Step 2 — Keep target columns only

In [ ]:
# These are the only three Census columns needed for this entire pipeline
# B08141_001E = Estimate: Total households in the tract
# B08141_002E = Estimate: Total: No vehicle available (zero vehicle households)
# B08141_002M = Margin of error for the zero vehicle count (90% confidence level)
TARGET_TOTAL = 'B08141_001E'
TARGET_ZERO  = 'B08141_002E'
TARGET_MOE   = 'B08141_002M'

cols_needed = ['GEO_ID', TARGET_TOTAL, TARGET_ZERO, TARGET_MOE]
df = df_raw[cols_needed].copy()

print(f"Columns before: {df_raw.shape[1]}")
print(f"Columns after:  {df.shape[1]}")
print()
print("Kept columns:")
print(df.dtypes)

In [ ]:
# Column metadata card — explains what each column means for the reader
col_meta = pd.DataFrame({
    'Census Code':    ['B08141_001E', 'B08141_002E', 'B08141_002M'],
    'Pipeline Name':  ['total_households', 'zero_vehicle_hh', 'moe_zero_vehicle'],
    'What it measures': [
        'Total households that commute to work',
        'Households with no privately owned vehicle',
        'Margin of error for zero-vehicle count (90% confidence)'
    ],
    'Role in pipeline': [
        'Denominator for rate computation',
        'Numerator for rate computation',
        'Precision weight in spatial join (1/MOE²)'
    ],
    'Columns dropped':  [f'{df_raw.shape[1] - 4} columns removed ({df_raw.shape[1]} → 4)', '', '']
})

(col_meta.style
 .set_caption(f'Table: Target Column Selection — 4 of {df_raw.shape[1]} Columns Retained')
 .set_properties(**{'text-align': 'left', 'padding': '6px 12px', 'font-size': '13px'})
 .set_table_styles([
     {'selector': 'caption', 'props': [('font-size','14px'),('font-weight','bold'),('padding','8px 0')]},
     {'selector': 'th', 'props': [('background-color','#534AB7'),('color','white'),('padding','7px 12px')]},
     {'selector': 'tr:nth-child(even)', 'props': [('background-color','#f5f4ff')]},
 ])
 .hide(axis='index'))


### Step 3 — Strip GEO_ID prefix and verify GEOID length

In [ ]:
# The Census prefix 1400000US must be stripped to match the shapefile GEOID format
# 140 = census tract summary level | 0000 = geographic variant | US = country code
# Example: '1400000US36005000100' → '36005000100'
df['GEOID'] = df['GEO_ID'].str.replace('1400000US', '', regex=False)

# Extract the county FIPS code from GEOID characters 2–4 (0-indexed)
# GEOID structure: [state 2 digits][county 3 digits][tract 6 digits] = 11 digits total
# Example: '36005000100' → positions [2:5] = '005' (Bronx County)
df['COUNTYFP'] = df['GEOID'].str[2:5]

# Zero-pad any GEOID shorter than 11 digits (safety net — Census GEOIDs are always 11 digits)
df['GEOID'] = df['GEOID'].str.zfill(11)

# Confirm every GEOID is exactly 11 digits
assert (df['GEOID'].str.len() == 11).all(), "Some GEOIDs are not 11 digits"

print(f"GEOID sample:          {df['GEOID'].head(3).tolist()}")
print(f"All exactly 11 digits: {(df['GEOID'].str.len() == 11).all()}")
print(f"Unique counties:       {df['COUNTYFP'].nunique()}")

total_skew = df[pd.to_numeric(df[TARGET_TOTAL], errors='coerce') > 0][TARGET_TOTAL].apply(pd.to_numeric).skew()
print(f"Total households skewness: {total_skew:.4f} (right-skewed — document in report)")

In [ ]:
# GEOID anatomy — before and after prefix strip
sample_raw   = '1400000US36005000100'
sample_clean = '36005000100'

geoid_table = pd.DataFrame({
    'Stage':         ['Raw GEO_ID (from CSV)', 'After stripping 1400000US'],
    'Example':       [sample_raw, sample_clean],
    'Length':        [len(sample_raw), len(sample_clean)],
    'State (chars 0-1)': ['N/A', sample_clean[0:2]],
    'County (chars 2-4)': ['N/A', sample_clean[2:5]],
    'Tract (chars 5-10)': ['N/A', sample_clean[5:11]],
    'Matches shapefile GEOID': ['✗ No', '✓ Yes']
})

assert (df['GEOID'].str.len() == 11).all(), "GEOID length check failed"
assert df['GEOID'].duplicated().sum() == 0, "Duplicate GEOIDs found after stripping"

# total_skew already computed in Step 3 (Cell 14) — reuse here
print(f"GEOID verification: all {df['GEOID'].nunique()} GEOIDs are exactly 11 digits ✓")
print(f"Total households skewness: {total_skew:.4f}  (right-skewed — document in limitations)")
print()

(geoid_table.style
 .set_caption('Table: GEO_ID Format Before and After Prefix Strip')
 .set_properties(**{'text-align': 'left', 'padding': '6px 12px', 'font-size': '13px'})
 .set_table_styles([
     {'selector': 'caption', 'props': [('font-size','14px'),('font-weight','bold'),('padding','8px 0')]},
     {'selector': 'th', 'props': [('background-color','#534AB7'),('color','white'),('padding','7px 12px')]},
     {'selector': 'tr:nth-child(1)', 'props': [('background-color','#ffeaea')]},
     {'selector': 'tr:nth-child(2)', 'props': [('background-color','#e8f5e9')]},
 ])
 .hide(axis='index'))


### Step 4 — Convert columns to numeric (defensive)

In [ ]:
# With skiprows=[1] these columns load as int64 already.
# This conversion step is written defensively to protect against future Census data releases
# that may introduce suppression codes such as -999 or blank strings.
# errors='coerce' converts any unreadable value to NaN instead of raising an error.
for col in [TARGET_TOTAL, TARGET_ZERO, TARGET_MOE]:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print("Data types after conversion:")
print(df[[TARGET_TOTAL, TARGET_ZERO, TARGET_MOE]].dtypes)
print()

# If any value was coerced to NaN it means a suppression code entered the data.
# That NaN would propagate silently through the rate formula and bias zone rates.
assert df[TARGET_TOTAL].isna().sum() == 0,     f"Non-numeric values in {TARGET_TOTAL}: {df[TARGET_TOTAL].isna().sum()} — check for ACS suppression codes"
assert df[TARGET_ZERO].isna().sum() == 0,     f"Non-numeric values in {TARGET_ZERO}: {df[TARGET_ZERO].isna().sum()} — check for ACS suppression codes"
assert df[TARGET_MOE].isna().sum() == 0,     f"Non-numeric values in {TARGET_MOE}: {df[TARGET_MOE].isna().sum()} — check for ACS suppression codes"

print(f"Non-numeric values in total:     {df[TARGET_TOTAL].isna().sum()}  ✓")
print(f"Non-numeric values in zero veh:  {df[TARGET_ZERO].isna().sum()}  ✓")
print(f"Non-numeric values in MOE:       {df[TARGET_MOE].isna().sum()}  ✓")

In [ ]:
# Dtype verification — confirms no suppression codes entered the data
dtype_check = pd.DataFrame({
    'Column':         ['B08141_001E (total_households)', 'B08141_002E (zero_vehicle_hh)', 'B08141_002M (moe_zero_vehicle)'],
    'Dtype after load':  [str(df_raw['B08141_001E'].dtype), str(df_raw['B08141_002E'].dtype), str(df_raw['B08141_002M'].dtype)],
    'NaN after coerce':  [int(df[TARGET_TOTAL].isna().sum()), int(df[TARGET_ZERO].isna().sum()), int(df[TARGET_MOE].isna().sum())],
    'Suppression codes': ['None detected ✓', 'None detected ✓', 'None detected ✓'],
    'Action':            ['Ready for computation', 'Ready for computation', 'Carry forward for precision weighting']
})

(dtype_check.style
 .set_caption('Table: Numeric Dtype Verification — Defensive Conversion Check')
 .set_properties(**{'text-align': 'left', 'padding': '6px 12px', 'font-size': '13px'})
 .set_table_styles([
     {'selector': 'caption', 'props': [('font-size','14px'),('font-weight','bold'),('padding','8px 0')]},
     {'selector': 'th', 'props': [('background-color','#534AB7'),('color','white'),('padding','7px 12px')]},
     {'selector': 'tr:nth-child(even)', 'props': [('background-color','#f5f4ff')]},
 ])
 .hide(axis='index'))


### Step 5 — CV check on margin of error and MOE decision

In [ ]:
# Coefficient of variation (CV) = standard Census Bureau measure for estimate reliability
# CV = MOE / (1.645 × estimate)
# CV > 0.5 means the margin of error exceeds 50% of the estimate — unreliable
df_cv = df[df[TARGET_TOTAL] > 0].copy()
df_cv['cv'] = df_cv[TARGET_MOE] / (1.645 * df_cv[TARGET_ZERO].replace(0, np.nan))

noisy    = (df_cv['cv'] > 0.5).sum()
reliable = (df_cv['cv'] <= 0.5).sum()
null_cv  = df_cv['cv'].isna().sum()

print(f"Reliable tracts (CV <= 0.5): {reliable}")
print(f"Noisy tracts    (CV >  0.5): {noisy}")
print(f"Tracts with CV = NaN:        {null_cv}  (zero vehicle count is zero — division undefined)")
print()

noisy_rate    = df_cv[df_cv['cv'] > 0.5][TARGET_ZERO].sum() / df_cv[df_cv['cv'] > 0.5][TARGET_TOTAL].sum()
reliable_rate = df_cv[df_cv['cv'] <= 0.5][TARGET_ZERO].sum() / df_cv[df_cv['cv'] <= 0.5][TARGET_TOTAL].sum()
print(f"Avg zero vehicle rate — noisy tracts:    {noisy_rate:.4f}")
print(f"Avg zero vehicle rate — reliable tracts: {reliable_rate:.4f}")
print()
print("Decision: keep all tracts.")
print("Dropping noisy tracts removes low-rate outer borough areas and biases every zone upward.")
print("The 1/MOE² precision weighting in Step 17 reduces their influence without removing them.")

In [ ]:
# Plot 1: Census tract statistical reliability by borough
df_cv = df[df[TARGET_TOTAL] > 0].copy()
df_cv['cv'] = df_cv[TARGET_MOE] / (1.645 * df_cv[TARGET_ZERO].replace(0, np.nan))
df_cv['borough'] = df_cv['COUNTYFP'].map(COUNTY_TO_BOROUGH)
df_cv['zero_vehicle_rate'] = (df_cv[TARGET_ZERO] / df_cv[TARGET_TOTAL]).clip(0,1)
df_cv['reliability'] = df_cv['cv'].apply(
    lambda x: 'Noisy (CV > 0.5)' if x > 0.5
    else ('CV=NaN (zero vehicles)' if pd.isna(x) else 'Reliable (CV ≤ 0.5)')
)

rel_counts = df_cv.groupby(['borough','reliability']).size().unstack(fill_value=0).reindex(BOROUGH_ORDER)
cols   = [c for c in ['Reliable (CV ≤ 0.5)', 'Noisy (CV > 0.5)', 'CV=NaN (zero vehicles)'] if c in rel_counts.columns]
colors_rc = ['#2E8B57', '#D85A30', '#888888']

fig, ax = plt.subplots(figsize=(9, 5))
bottom = np.zeros(len(rel_counts))
for col, col_c in zip(cols, colors_rc):
    vals = rel_counts[col].values
    ax.bar(BOROUGH_ORDER, vals, bottom=bottom, label=col, color=col_c, alpha=0.88, edgecolor='white', linewidth=0.5)
    for i, (h, b) in enumerate(zip(vals, bottom)):
        if h > 8:
            ax.text(i, b + h/2, str(int(h)), ha='center', va='center', fontsize=9, color='white', fontweight='bold')
    bottom += vals

ax.set_xlabel('Borough')
ax.set_ylabel('Number of census tracts')
ax.set_title('Figure: Census Tract Statistical Reliability by Borough\n(CV = Margin of Error ÷ (1.645 × Estimate))', pad=10)
ax.legend(loc='upper right', framealpha=0.9)
ax.set_ylim(0, bottom.max() * 1.12)
plt.tight_layout()
plt.show()
print("Figure saved. Staten Island has the highest proportion of noisy tracts (57%).")


In [ ]:
# Plot 2: Zero vehicle rate distribution by reliability category
df_cv2 = df_cv[df_cv['reliability'] != 'CV=NaN (zero vehicles)'].copy()
order_rel = ['Reliable (CV ≤ 0.5)', 'Noisy (CV > 0.5)']
colors_v  = ['#2E8B57', '#D85A30']

fig, ax = plt.subplots(figsize=(7, 5))
sns.violinplot(data=df_cv2, x='reliability', y='zero_vehicle_rate',
               order=order_rel, palette=colors_v, ax=ax, inner='box', cut=0, linewidth=1.2)

means_rel = df_cv2.groupby('reliability')['zero_vehicle_rate'].mean()
for i, cat in enumerate(order_rel):
    ax.text(i, means_rel[cat] + 0.04, f"mean = {means_rel[cat]:.1%}", ha='center',
            fontsize=10, fontweight='bold', color=colors_v[i])

ax.set_xlabel('Tract reliability category')
ax.set_ylabel('Zero vehicle rate')
ax.set_title('Figure: Zero Vehicle Rate — Reliable vs Noisy Tracts\nDropping noisy tracts would bias zone rates upward by ~35 percentage points', pad=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.set_ylim(-0.05, 1.15)
plt.tight_layout()
plt.show()


In [ ]:
# CV summary table
noisy_rate    = df_cv[df_cv['cv'] > 0.5][TARGET_ZERO].sum() / df_cv[df_cv['cv'] > 0.5][TARGET_TOTAL].sum()
reliable_rate = df_cv[df_cv['cv'] <= 0.5][TARGET_ZERO].sum() / df_cv[df_cv['cv'] <= 0.5][TARGET_TOTAL].sum()

cv_summary = pd.DataFrame({
    'Category':        ['Reliable (CV ≤ 0.5)', 'Noisy (CV > 0.5)', 'CV = NaN (zero vehicles)'],
    'Tract count':     [int((df_cv['cv'] <= 0.5).sum()), int((df_cv['cv'] > 0.5).sum()), int(df_cv['cv'].isna().sum())],
    'Avg zero veh rate': [f'{reliable_rate:.1%}', f'{noisy_rate:.1%}', 'N/A (rate = 0.0)'],
    'Decision':        ['Include — standard weighting', 'Include — 1/MOE² precision weighting', 'Include — rate = 0.0 (Type 2)']
})

(cv_summary.style
 .set_caption('Table: CV Check Summary — All 2,327 Tracts Retained (No Dropping)')
 .set_properties(**{'text-align': 'left', 'padding': '6px 12px', 'font-size': '13px'})
 .set_table_styles([
     {'selector': 'caption', 'props': [('font-size','14px'),('font-weight','bold'),('padding','8px 0')]},
     {'selector': 'th', 'props': [('background-color','#2E8B57'),('color','white'),('padding','7px 12px')]},
     {'selector': 'tr:nth-child(even)', 'props': [('background-color','#f0f9f4')]},
 ])
 .hide(axis='index'))


### Step 6 — Drop zero and null total_households

In [ ]:
rows_before = len(df)

# Type 1 tracts (total_households = 0): parks, airports, commercial areas with no residents
# NaN tracts: rows where numeric conversion produced a missing value
# Both are removed here — division by zero gives an undefined rate
df_valid = df[(df[TARGET_TOTAL] > 0) & (df[TARGET_TOTAL].notna())].copy()

removed = rows_before - len(df_valid)

print(f"Rows before filter: {rows_before}")
print(f"Rows removed:       {removed}  (Type 1 zeros and NaN values)")
print(f"Rows remaining:     {len(df_valid)}")
print()

# Type 2 tracts: zero_vehicle_hh = 0 but total_households > 0
# These are genuine residential areas where every household owns a car — keep them
type2 = (df_valid[TARGET_ZERO] == 0).sum()
print(f"Type 2 tracts with zero_vehicle_hh = 0 (genuine all-car areas, kept): {type2}")
print()
print(f"NOTE: The {removed} removed Type 1 tracts produce NaN in the shapefile merge (Step 12)")
print(f"and are dropped by dropna before the overlay (Step 15). They never reach Step 19.")
print(f"The NaN zones filled in Step 19 are island zones with zero overlay fragments — a separate cause.")

In [ ]:
# Plot 3: Three zero types per borough
# Type 1 tracts (total_households = 0) are NOT in df_cv or df_valid — both filter total > 0
# Must use df (all 2327 tracts including Type 1 zeros) vs df_valid (2228 residential)
type1_counts = (
    df['COUNTYFP'].map(COUNTY_TO_BOROUGH).value_counts() -
    df_valid['COUNTYFP'].map(COUNTY_TO_BOROUGH).value_counts()
).reindex(BOROUGH_ORDER, fill_value=0).clip(lower=0)
type2_counts = df_valid[df_valid[TARGET_ZERO] == 0]['COUNTYFP'].map(COUNTY_TO_BOROUGH).value_counts().reindex(BOROUGH_ORDER, fill_value=0)
valid_counts = df_valid[df_valid[TARGET_ZERO] > 0]['COUNTYFP'].map(COUNTY_TO_BOROUGH).value_counts().reindex(BOROUGH_ORDER, fill_value=0)

x, w = np.arange(len(BOROUGH_ORDER)), 0.26
fig, ax = plt.subplots(figsize=(10, 5))
b1 = ax.bar(x - w, valid_counts,  w, label='Valid residential (used in pipeline)', color='#2E8B57', alpha=0.88, edgecolor='white')
b2 = ax.bar(x,     type2_counts,  w, label='Type 2: rate = 0.0 (all households own a car)', color='#534AB7', alpha=0.88, edgecolor='white')
b3 = ax.bar(x + w, type1_counts,  w, label='Type 1: dropped (parks / airports / water)', color='#D85A30', alpha=0.88, edgecolor='white')

for bars in [b1, b2, b3]:
    for bar in bars:
        h = bar.get_height()
        if h > 0:
            ax.text(bar.get_x() + bar.get_width()/2, h + 1.5, int(h), ha='center', va='bottom', fontsize=8.5)

ax.set_xticks(x); ax.set_xticklabels(BOROUGH_ORDER)
ax.set_ylabel('Number of census tracts')
ax.set_title('Figure: Census Tract Classification by Zero-Value Type\nThree distinct zero types require different treatment', pad=10)
ax.legend(loc='upper right', framealpha=0.9, fontsize=9)
plt.tight_layout()
plt.show()


In [ ]:
# Zero types summary table
zero_types_table = pd.DataFrame({
    'Zero type':    ['Type 1 — total_households = 0', 'Type 2 — zero_vehicle_hh = 0 (total > 0)', 'Type 3 — NaN after spatial join'],
    'Real-world meaning': ['Parks, airports, commercial areas with no residents', 'Residential area where every household owns a car', 'Island zones with no census tract overlap'],
    'Count':        [f"{rows_before - len(df_valid)} tracts", f"{int((df_valid[TARGET_ZERO] == 0).sum())} tracts", "4 zones (after overlay)"],
    'Action':       ['DROP — 0 ÷ 0 is undefined', 'KEEP — rate = 0.0 is genuine signal', 'FILL with borough mean (Phase 3 Step 19)'],
    'Cause':        ['Zero denominator', 'Zero numerator only', 'No residential population in zone']
})

print(f"Rows before filter: {rows_before}")
print(f"Rows removed (Type 1 + NaN): {rows_before - len(df_valid)}")
print(f"Rows remaining:     {len(df_valid)}")
print()
print("NOTE: Type 1 NaN (from shapefile merge in Step 12) is separate from")
print("Type 3 NaN (from zero overlay fragments in Step 15). dropna before")
print("the overlay (Step 15) ensures these two causes never mix.")

(zero_types_table.style
 .set_caption('Table: Three Zero-Value Types and Their Treatment')
 .set_properties(**{'text-align': 'left', 'padding': '6px 12px', 'font-size': '13px'})
 .set_table_styles([
     {'selector': 'caption', 'props': [('font-size','14px'),('font-weight','bold'),('padding','8px 0')]},
     {'selector': 'th', 'props': [('background-color','#D85A30'),('color','white'),('padding','7px 12px')]},
     {'selector': 'tr:nth-child(1)', 'props': [('background-color','#ffeaea')]},
     {'selector': 'tr:nth-child(2)', 'props': [('background-color','#e8f5e9')]},
     {'selector': 'tr:nth-child(3)', 'props': [('background-color','#fff8e1')]},
 ])
 .hide(axis='index'))


### Step 7 — Rename columns to readable names

In [ ]:
df_valid = df_valid.rename(columns={
    TARGET_TOTAL: 'total_households',
    TARGET_ZERO:  'zero_vehicle_hh',
    TARGET_MOE:   'moe_zero_vehicle'
})

# Drop GEO_ID — the original prefixed string (1400000US...) is no longer needed
# GEOID (stripped, zero-padded) is the join key used downstream
df_valid.drop(columns=['GEO_ID'], inplace=True)

print("Final Phase 1 columns:")
print(df_valid.columns.tolist())
print()
print("Sample rows:")
print(df_valid[['GEOID', 'total_households', 'zero_vehicle_hh', 'moe_zero_vehicle']].head(5))

In [ ]:
# Column rename reference table
rename_table = pd.DataFrame({
    'Census Code':      ['B08141_001E', 'B08141_002E', 'B08141_002M'],
    'Pipeline Name':    ['total_households', 'zero_vehicle_hh', 'moe_zero_vehicle'],
    'Meaning':          ['Total commuting households in the tract', 'Households with zero privately owned vehicles', 'Margin of error for zero-vehicle count'],
    'Dropped columns':  ['GEO_ID (replaced by GEOID)', '', '']
})

(rename_table.style
 .set_caption('Table: Column Renaming — Census Codes to Readable Names')
 .set_properties(**{'text-align': 'left', 'padding': '6px 12px', 'font-size': '13px'})
 .set_table_styles([
     {'selector': 'caption', 'props': [('font-size','14px'),('font-weight','bold'),('padding','8px 0')]},
     {'selector': 'th', 'props': [('background-color','#534AB7'),('color','white'),('padding','7px 12px')]},
     {'selector': 'tr:nth-child(even)', 'props': [('background-color','#f5f4ff')]},
 ])
 .hide(axis='index'))


---
## Phase 2: Compute Rate and Validation

### Step 8 — Compute zero_vehicle_rate

In [ ]:
# zero_vehicle_rate = proportion of households with no privately owned vehicle
# Range: 0.0 (every household owns a car) to 1.0 (no household owns a car)
# Higher rate → more transit-dependent zone → higher expected taxi demand
df_valid['zero_vehicle_rate'] = df_valid['zero_vehicle_hh'] / df_valid['total_households']

print("Zero vehicle rate statistics:")
print(df_valid['zero_vehicle_rate'].describe().round(4))
print()

# Map COUNTYFP to borough names using COUNTY_TO_BOROUGH derived from FIPS file
df_valid['borough'] = df_valid['COUNTYFP'].map(COUNTY_TO_BOROUGH)

print("Average rate per borough:")
print(df_valid.groupby('borough')['zero_vehicle_rate'].mean().round(4))
print()

# Compute BOROUGH_MEANS from actual data — used to fill NaN zones in Step 19
# Borough mean is more accurate than citywide mean (up to 62pp difference across boroughs)
BOROUGH_MEANS = (
    df_valid.groupby('borough')['zero_vehicle_rate']
    .mean()
    .round(4)
    .to_dict()
)
assert len(BOROUGH_MEANS) == 5, \
    f"Expected 5 boroughs in BOROUGH_MEANS, got {len(BOROUGH_MEANS)}: {list(BOROUGH_MEANS.keys())}"

print("Borough means stored for NaN fill in Step 19:")
print(BOROUGH_MEANS)

In [ ]:
# Plot 4: Zero vehicle rate distribution by borough (tract level)
fig, ax = plt.subplots(figsize=(10, 5))
palette = [BOROUGH_COLORS[b] for b in BOROUGH_ORDER]
sns.violinplot(data=df_valid, x='borough', y='zero_vehicle_rate',
               order=BOROUGH_ORDER, palette=palette, ax=ax, inner='box', cut=0, linewidth=1.2)
for i, b in enumerate(BOROUGH_ORDER):
    m = BOROUGH_MEANS[b]
    ax.text(i, m + 0.04, f'{m:.1%}', ha='center', fontsize=9.5,
            fontweight='bold', color=BOROUGH_COLORS[b])
ax.set_xlabel('Borough')
ax.set_ylabel('Zero vehicle rate (tract level)')
ax.set_title('Figure: Zero Vehicle Rate Distribution by Borough — Tract Level\n2,228 residential tracts after Type 1 removal', pad=10)
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.set_ylim(-0.05, 1.15)
plt.tight_layout()
plt.show()


In [ ]:
# Borough means summary table
bm_stats = df_valid.groupby('borough')['zero_vehicle_rate'].agg(
    tract_count='count', mean='mean', std='std', min='min', max='max'
).reindex(BOROUGH_ORDER).round(4).reset_index()
bm_stats.columns = ['Borough', 'Tract Count', 'Mean Rate', 'Std Dev', 'Min Rate', 'Max Rate']
bm_stats['Mean Rate'] = bm_stats['Mean Rate'].map('{:.1%}'.format)
bm_stats['Std Dev']   = bm_stats['Std Dev'].map('{:.1%}'.format)
bm_stats['Min Rate']  = bm_stats['Min Rate'].map('{:.1%}'.format)
bm_stats['Max Rate']  = bm_stats['Max Rate'].map('{:.1%}'.format)
bm_stats['Role'] = ['Fill value for island zones (NaN fill Step 19)'] + [''] * 4

(bm_stats.style
 .set_caption('Table: Borough Mean Zero Vehicle Rates — Computed from df_valid (Used in Step 19 NaN Fill)')
 .set_properties(**{'text-align': 'left', 'padding': '6px 12px', 'font-size': '13px'})
 .set_table_styles([
     {'selector': 'caption', 'props': [('font-size','14px'),('font-weight','bold'),('padding','8px 0')]},
     {'selector': 'th', 'props': [('background-color','#534AB7'),('color','white'),('padding','7px 12px')]},
     {'selector': 'tr:nth-child(odd)', 'props': [('background-color','#f5f4ff')]},
 ])
 .hide(axis='index'))


### Step 9 — Range validation and clip to [0, 1]

In [ ]:
# Clip to [0, 1] to correct any floating point rounding errors
# (e.g. a value of 1.0000001 from internal Census rounding)
df_valid['zero_vehicle_rate'] = df_valid['zero_vehicle_rate'].clip(0, 1)

assert df_valid['zero_vehicle_rate'].between(0, 1).all(), "Rate out of [0, 1] range after clip"

print(f"Min rate after clip: {df_valid['zero_vehicle_rate'].min():.6f}")
print(f"Max rate after clip: {df_valid['zero_vehicle_rate'].max():.6f}")
print(f"All rates in [0, 1]: {df_valid['zero_vehicle_rate'].between(0, 1).all()}")

In [ ]:
# Range check table — confirms clip is a safety net, not masking real errors
pre_clip_min  = (df_valid['zero_vehicle_hh'] / df_valid['total_households']).min()
pre_clip_max  = (df_valid['zero_vehicle_hh'] / df_valid['total_households']).max()
out_of_range  = int(((df_valid['zero_vehicle_hh'] / df_valid['total_households']) < 0).sum() +
                    ((df_valid['zero_vehicle_hh'] / df_valid['total_households']) > 1).sum())

range_table = pd.DataFrame({
    'Stage':            ['Before clip', 'After clip'],
    'Min rate':         [f'{pre_clip_min:.6f}', f'{df_valid["zero_vehicle_rate"].min():.6f}'],
    'Max rate':         [f'{pre_clip_max:.6f}', f'{df_valid["zero_vehicle_rate"].max():.6f}'],
    'Values outside [0,1]': [out_of_range, 0],
    'Action':           ['Check', 'Assert ✓']
})

(range_table.style
 .set_caption('Table: Range Validation — Clip Corrects Floating Point Rounding Errors')
 .set_properties(**{'text-align': 'left', 'padding': '6px 12px', 'font-size': '13px'})
 .set_table_styles([
     {'selector': 'caption', 'props': [('font-size','14px'),('font-weight','bold'),('padding','8px 0')]},
     {'selector': 'th', 'props': [('background-color','#534AB7'),('color','white'),('padding','7px 12px')]},
     {'selector': 'tr:nth-child(even)', 'props': [('background-color','#f5f4ff')]},
 ])
 .hide(axis='index'))


### Step 10 — Duplicate GEOID assertion

In [ ]:
duplicate_count = df_valid['GEOID'].duplicated().sum()
assert duplicate_count == 0, f"{duplicate_count} duplicate GEOIDs found"

print(f"Duplicate GEOIDs: {duplicate_count}")
print(f"Total unique GEOIDs: {df_valid['GEOID'].nunique()}")
print("All GEOIDs are unique. Safe to proceed with shapefile merge.")

In [ ]:
# GEOID uniqueness validation table
uniq_table = pd.DataFrame({
    'Check':       ['Total GEOIDs', 'Unique GEOIDs', 'Duplicate GEOIDs', 'Safe to merge'],
    'Value':       [len(df_valid), df_valid['GEOID'].nunique(), int(df_valid['GEOID'].duplicated().sum()), 'Yes ✓'],
    'Expected':    [2228, 2228, 0, 'Yes ✓'],
    'Status':      ['✓ Pass', '✓ Pass', '✓ Pass', '✓ Pass']
})

(uniq_table.style
 .set_caption('Table: GEOID Uniqueness Check — No Duplicate Census Tracts')
 .set_properties(**{'text-align': 'left', 'padding': '6px 12px', 'font-size': '13px'})
 .set_table_styles([
     {'selector': 'caption', 'props': [('font-size','14px'),('font-weight','bold'),('padding','8px 0')]},
     {'selector': 'th', 'props': [('background-color','#2E8B57'),('color','white'),('padding','7px 12px')]},
     {'selector': 'tr:nth-child(even)', 'props': [('background-color','#f0f9f4')]},
 ])
 .hide(axis='index'))


---
## Phase 3: Spatial Join

### Step 11 — Taxi zone EDA and geometry audit

In [ ]:
taxi_gdf    = gpd.read_file(TAXI_SHP)
taxi_lookup = pd.read_csv(LOOKUP_PATH)

# CRS assertion — confirms shapefile is EPSG:2263 before any spatial work
assert taxi_gdf.crs.to_epsg() == 2263,     f"Expected EPSG:2263 but got {taxi_gdf.crs}. Reproject taxi_gdf before proceeding."

# Geometry integrity — invalid or empty geometries corrupt the spatial overlay
# and produce wrong fragment areas that silently bias zone rates
assert (~taxi_gdf.geometry.is_valid).sum() == 0,     f"{(~taxi_gdf.geometry.is_valid).sum()} invalid taxi zone geometries found"
assert taxi_gdf.geometry.is_empty.sum() == 0,     f"{taxi_gdf.geometry.is_empty.sum()} empty taxi zone geometries found"
assert taxi_gdf['LocationID'].duplicated().sum() == 0,     f"{taxi_gdf['LocationID'].duplicated().sum()} duplicate LocationIDs in taxi zones"

print(f"Taxi zones loaded:          {len(taxi_gdf)}")
print(f"CRS:                        {taxi_gdf.crs}  ✓")
print(f"Invalid geometries:         {(~taxi_gdf.geometry.is_valid).sum()}  ✓")
print(f"Empty geometries:           {taxi_gdf.geometry.is_empty.sum()}  ✓")
print(f"Duplicate LocationIDs:      {taxi_gdf['LocationID'].duplicated().sum()}  ✓")
print(f"LocationID range:           {taxi_gdf['LocationID'].min()} to {taxi_gdf['LocationID'].max()}")
print()
print("Zones per borough:")
print(taxi_gdf['borough'].value_counts())
print()

lookup_only = set(taxi_lookup['LocationID']) - set(taxi_gdf['LocationID'])
print(f"LocationIDs in lookup but not shapefile: {sorted(lookup_only)}")
print("These are Unknown (264) and Outside of NYC (265) — lookup only, not in spatial data")
print()

ewr = taxi_gdf[taxi_gdf['LocationID'] == 1][['LocationID', 'zone', 'borough']]
print("LocationID 1 details:")
print(ewr.to_string(index=False))
print("Newark Airport sits in New Jersey — no NYC census tract covers it.")
print("It will produce zero fragments in the overlay → NaN → excluded in Step 20.")

In [ ]:
# Taxi zone summary table
zone_by_borough = taxi_gdf['borough'].value_counts().reset_index()
zone_by_borough.columns = ['Borough', 'Zone Count']
zone_by_borough['% of Total'] = (zone_by_borough['Zone Count'] / zone_by_borough['Zone Count'].sum() * 100).round(1).astype(str) + '%'
zone_by_borough['Geometry types'] = zone_by_borough['Borough'].map(
    lambda b: str(taxi_gdf[taxi_gdf['borough']==b].geometry.geom_type.value_counts().to_dict())
)
zone_by_borough.loc[len(zone_by_borough)] = ['TOTAL', zone_by_borough['Zone Count'].sum(), '100%', '']

lookup_only = sorted(set(taxi_lookup['LocationID']) - set(taxi_gdf['LocationID']))
print(f"Shapefile: 263 zones, CRS: EPSG:2263 ✓, 0 invalid geometries ✓")
print(f"LocationIDs in lookup only (not in shapefile): {lookup_only}  — filtered from lookup CSV only")
print()

(zone_by_borough.style
 .set_caption('Table: NYC Taxi Zone Count by Borough (TLC Shapefile)')
 .set_properties(**{'text-align': 'left', 'padding': '6px 12px', 'font-size': '13px'})
 .set_table_styles([
     {'selector': 'caption', 'props': [('font-size','14px'),('font-weight','bold'),('padding','8px 0')]},
     {'selector': 'th', 'props': [('background-color','#1D9E75'),('color','white'),('padding','7px 12px')]},
     {'selector': 'tr:last-child', 'props': [('font-weight','bold'),('background-color','#e8f5e9')]},
 ])
 .hide(axis='index'))


In [ ]:
# Taxi zone map coloured by borough
taxi_display = taxi_gdf.to_crs(epsg=4326)
borough_col_map = {**BOROUGH_COLORS, 'EWR': '#888888'}
taxi_display['color'] = taxi_display['borough'].map(borough_col_map)

fig, ax = plt.subplots(figsize=(9, 8))
for borough in BOROUGH_ORDER + ['EWR']:
    subset = taxi_display[taxi_display['borough'] == borough]
    if len(subset) > 0:
        subset.plot(ax=ax, color=borough_col_map.get(borough, '#888888'),
                    alpha=0.75, edgecolor='white', linewidth=0.3, label=borough)

ax.set_title('Figure: NYC TLC Taxi Zones (263 zones coloured by borough)\nSource: NYC TLC Taxi Zone Shapefile, EPSG:2263 reprojected to EPSG:4326', pad=10)
ax.legend(loc='upper left', fontsize=9, framealpha=0.9)
ax.set_axis_off()
plt.tight_layout()
plt.show()


### Step 12 — Load census tract shapefile, filter to NYC, and merge rate

In [ ]:
tract_raw = gpd.read_file(TRACT_SHP)
print(f"NY State tracts in shapefile: {len(tract_raw)}")

tract_gdf = tract_raw[tract_raw['COUNTYFP'].isin(NYC_COUNTIES)].copy()
print(f"NYC tracts after filter:      {len(tract_gdf)}")
print()
print("Tract count per NYC county:")
print(tract_gdf.groupby('COUNTYFP')['GEOID'].count())

census_data = df_valid[['GEOID', 'zero_vehicle_rate', 'moe_zero_vehicle']].copy()

# Left merge: shapefile is the left table so all 2327 NYC shapefile tracts are kept
# Tracts removed in Step 6 (Type 1 zeros, total_households = 0) produce NaN here
tract_gdf = tract_gdf.merge(census_data, on='GEOID', how='left')
tract_gdf = tract_gdf[['GEOID', 'COUNTYFP', 'zero_vehicle_rate', 'moe_zero_vehicle', 'geometry']]

matched   = tract_gdf['zero_vehicle_rate'].notna().sum()
unmatched = tract_gdf['zero_vehicle_rate'].isna().sum()

# Every residential tract in df_valid must have a matching GEOID in the shapefile.
# A mismatch means GEOIDs are formatted differently between the two sources — 
# those tracts would be silently dropped from the spatial aggregation, biasing zone rates.
assert matched == len(df_valid),     f"GEOID mismatch: {matched} tracts matched but expected {len(df_valid)} — verify GEOID stripping in Cell 11"
assert unmatched == (rows_before - len(df_valid)),     f"Unexpected NaN count: {unmatched} but expected {rows_before - len(df_valid)} Type 1 zero tracts"

print()
print(f"Tracts with rate matched:  {matched}  ✓  (all {len(df_valid)} residential tracts accounted for)")
print(f"Tracts with NaN rate:      {unmatched}  ✓  (exactly the {rows_before - len(df_valid)} Type 1 zero-hh tracts)")
print()
print(f"Tract CRS: {tract_gdf.crs}")

In [ ]:
# Census tract count per county — before and after NYC filter
all_counties = tract_raw['COUNTYFP'].value_counts().reset_index()
all_counties.columns = ['COUNTYFP', 'NY State Tract Count']

nyc_fips_ref = nyc_fips[['COUNTYFP', 'COUNTYNAME', 'borough_name']].copy()
nyc_fips_ref.columns = ['COUNTYFP', 'County Name', 'Borough']
tract_counts_nyc = tract_gdf.groupby('COUNTYFP').size().reset_index()
tract_counts_nyc.columns = ['COUNTYFP', 'NYC Tract Count']
tract_counts_nyc['Matched with rate'] = [
    int(tract_gdf[tract_gdf['COUNTYFP']==c]['zero_vehicle_rate'].notna().sum())
    for c in tract_counts_nyc['COUNTYFP']
]
tract_counts_nyc['Type 1 NaN (expected)'] = tract_counts_nyc['NYC Tract Count'] - tract_counts_nyc['Matched with rate']
tract_counts_nyc = nyc_fips_ref.merge(tract_counts_nyc, on='COUNTYFP', how='left')

print(f"NY State total: {len(tract_raw)} tracts  →  NYC only: {len(tract_gdf)} tracts (filter saves 2.3× overlay time)")
print(f"GEOID match: {int(tract_gdf['zero_vehicle_rate'].notna().sum())}/{len(df_valid)} = 100% ✓")
print()

(tract_counts_nyc.style
 .set_caption('Table: Census Tract Counts — NY State Filter to NYC (5,411 → 2,327 tracts)')
 .set_properties(**{'text-align': 'left', 'padding': '6px 12px', 'font-size': '13px'})
 .set_table_styles([
     {'selector': 'caption', 'props': [('font-size','14px'),('font-weight','bold'),('padding','8px 0')]},
     {'selector': 'th', 'props': [('background-color','#534AB7'),('color','white'),('padding','7px 12px')]},
     {'selector': 'tr:nth-child(even)', 'props': [('background-color','#f5f4ff')]},
 ])
 .hide(axis='index'))

# Drop COUNTYFP now that the table is built — not needed downstream
tract_gdf.drop(columns=['COUNTYFP'], inplace=True)
print(f"COUNTYFP dropped. tract_gdf columns: {tract_gdf.columns.tolist()}")

In [ ]:
# Census tract map coloured by borough
tract_display = tract_gdf.to_crs(epsg=4326)
tract_display['borough'] = tract_display['GEOID'].str[2:5].map(COUNTY_TO_BOROUGH)

fig, ax = plt.subplots(figsize=(9, 8))
for borough in BOROUGH_ORDER:
    subset = tract_display[tract_display['borough'] == borough]
    subset.plot(ax=ax, color=BOROUGH_COLORS[borough], alpha=0.65,
                edgecolor='white', linewidth=0.1, label=borough)

ax.set_title('Figure: NYC Census Tracts from TIGER Shapefile (2,327 tracts coloured by borough)\nNote: Tract boundaries do not align with taxi zone boundaries — spatial join needed', pad=10)
ax.legend(loc='upper left', fontsize=9, framealpha=0.9)
ax.set_axis_off()
plt.tight_layout()
plt.show()


### Step 13 — Slim taxi GeoDataFrame and compute zone_area for QA

In [ ]:
# Keep only the three columns needed from the taxi zone shapefile
taxi_gdf = taxi_gdf[['LocationID', 'borough', 'geometry']].copy()

# Compute zone_area BEFORE the overlay clips the polygon geometry
# zone_area is used ONLY in the coverage diagnostic (Step 16)
# It is NOT part of the precision weighting formula — see Step 17 for explanation
taxi_gdf['zone_area'] = taxi_gdf.geometry.area

print(f"Taxi zone CRS: {taxi_gdf.crs}")
print(f"Smallest zone: {taxi_gdf['zone_area'].min():>15,.0f} sq ft")
print(f"Largest zone:  {taxi_gdf['zone_area'].max():>15,.0f} sq ft")
print(f"Size ratio:    {taxi_gdf['zone_area'].max() / taxi_gdf['zone_area'].min():.0f}x larger")
print()
print("zone_area is used for the coverage diagnostic in Step 16 only.")
print("It is NOT included in the weighting formula — see Step 17 for the proof.")

In [ ]:
# Plot 6: Log-scale zone area distribution
fig, ax = plt.subplots(figsize=(8, 5))
log_areas = np.log10(taxi_gdf['zone_area'])
ax.hist(log_areas, bins=30, color='#534AB7', alpha=0.75, edgecolor='white', linewidth=0.5)
ax.axvline(np.log10(taxi_gdf['zone_area'].min()), color='#D85A30', lw=2, ls='--',
           label=f"Smallest: {taxi_gdf['zone_area'].min():,.0f} ft²")
ax.axvline(np.log10(taxi_gdf['zone_area'].max()), color='#1D9E75', lw=2, ls='--',
           label=f"Largest: {taxi_gdf['zone_area'].max():,.0f} ft²")
ax.set_xlabel('Zone area (log₁₀ scale, sq ft)')
ax.set_ylabel('Number of taxi zones')
ax.set_title(f'Figure: NYC Taxi Zone Area Distribution (log scale)\n{taxi_gdf["zone_area"].max()/taxi_gdf["zone_area"].min():.0f}× size difference justifies area weighting over simple average', pad=10)
ax.legend(framealpha=0.9)
plt.tight_layout()
plt.show()

zone_area_stats = pd.DataFrame({
    'Metric': ['Smallest zone', 'Largest zone', 'Mean zone area', 'Median zone area', 'Size ratio (max/min)'],
    'Value':  [
        f"{taxi_gdf['zone_area'].min():,.0f} ft²",
        f"{taxi_gdf['zone_area'].max():,.0f} ft²",
        f"{taxi_gdf['zone_area'].mean():,.0f} ft²",
        f"{taxi_gdf['zone_area'].median():,.0f} ft²",
        f"{taxi_gdf['zone_area'].max()/taxi_gdf['zone_area'].min():.0f}×"
    ],
    'Implication': [
        'Small Manhattan street zone',
        'Large Staten Island/Queens park zone',
        '—', '—',
        'A flat average would over-weight large zones — area weighting is essential'
    ]
})

print("zone_area is computed here (before overlay clips the geometry)")
print("It is used ONLY for the coverage diagnostic in Step 16.")
print("It is NOT part of the precision weighting formula — see Step 17 for the mathematical proof.")
print()

(zone_area_stats.style
 .set_caption('Table: NYC Taxi Zone Area Statistics (EPSG:2263, square feet)')
 .set_properties(**{'text-align': 'left', 'padding': '6px 12px', 'font-size': '13px'})
 .set_table_styles([
     {'selector': 'caption', 'props': [('font-size','14px'),('font-weight','bold'),('padding','8px 0')]},
     {'selector': 'th', 'props': [('background-color','#534AB7'),('color','white'),('padding','7px 12px')]},
     {'selector': 'tr:nth-child(even)', 'props': [('background-color','#f5f4ff')]},
 ])
 .hide(axis='index'))


### Step 14 — Reproject census tract shapefile to EPSG:2263

In [ ]:
# Both shapefiles must share the same CRS before any spatial operation
# Tract shapefile uses EPSG:4269 (NAD83) — different from EPSG:4326 (WGS84)
# both use degrees but they reference different datums
# Only the tract needs reprojecting — taxi zones are already confirmed EPSG:2263 (Step 11)
tract_gdf = tract_gdf.to_crs(epsg=2263)

assert (~tract_gdf.geometry.is_valid).sum() == 0,     f"{(~tract_gdf.geometry.is_valid).sum()} invalid geometries after reprojection"

# CRS match must be asserted before the overlay — a mismatch produces geometrically wrong
# fragment areas that silently bias all zone rates with no error raised
assert tract_gdf.crs == taxi_gdf.crs,     f"CRS mismatch before overlay: tracts={tract_gdf.crs}, zones={taxi_gdf.crs}"

print(f"Tract CRS after reproject:   {tract_gdf.crs}")
print(f"Taxi zone CRS:               {taxi_gdf.crs}")
print(f"CRS match:                   {tract_gdf.crs == taxi_gdf.crs}  ✓")
print(f"Invalid geometries:          {(~tract_gdf.geometry.is_valid).sum()}  ✓")

In [ ]:
# CRS alignment table
crs_table = pd.DataFrame({
    'Dataset':    ['Census tract shapefile (before)', 'Census tract shapefile (after reproject)', 'Taxi zone shapefile'],
    'CRS':        [str(tract_raw.crs), str(tract_gdf.crs), str(taxi_gdf.crs)],
    'EPSG Code':  ['4269', '2263', '2263'],
    'System':     ['NAD83 (geographic, degrees)', 'NY State Plane (projected, feet)', 'NY State Plane (projected, feet)'],
    'Match':      ['✗ No (cannot run overlay)', '✓ Yes (ready for overlay)', '✓ Reference']
})

(crs_table.style
 .set_caption('Table: Coordinate Reference System Alignment Before and After Reprojection')
 .set_properties(**{'text-align': 'left', 'padding': '6px 12px', 'font-size': '13px'})
 .set_table_styles([
     {'selector': 'caption', 'props': [('font-size','14px'),('font-weight','bold'),('padding','8px 0')]},
     {'selector': 'th', 'props': [('background-color','#534AB7'),('color','white'),('padding','7px 12px')]},
     {'selector': 'tr:nth-child(1)', 'props': [('background-color','#ffeaea')]},
     {'selector': 'tr:nth-child(2)', 'props': [('background-color','#e8f5e9')]},
     {'selector': 'tr:nth-child(3)', 'props': [('background-color','#e8f5e9')]},
 ])
 .hide(axis='index'))


### Step 15 — Run spatial intersection with gpd.overlay()

In [ ]:
# Drop tracts with NaN rate (Type 1 zero-household tracts) before the overlay
# This prevents their zero-area fragments from mixing with Type 3 NaN zones
tract_with_rate = tract_gdf.dropna(subset=['zero_vehicle_rate']).copy()
print(f"Tracts entering overlay: {len(tract_with_rate)}  (residential only)")
print()
print("Running spatial intersection — this may take 1 to 5 minutes...")

# Each row in the output is one fragment where a taxi zone and tract polygon physically overlap
# zone_area is NOT passed into the overlay — it is a QA-only attribute
overlay = gpd.overlay(
    taxi_gdf[['LocationID', 'geometry']],
    tract_with_rate[['GEOID', 'zero_vehicle_rate', 'moe_zero_vehicle', 'geometry']],
    how='intersection'
)

overlay['frag_area'] = overlay.geometry.area

print(f"Overlay complete.")
print(f"Total fragments produced: {len(overlay)}")
print()
print("Sample output:")
print(overlay[['LocationID', 'GEOID', 'zero_vehicle_rate']].head(5))

In [ ]:
# Overlay summary — immediately after overlay completes
overlay_summary = pd.DataFrame({
    'Metric':  [
        'Tracts before dropna (NYC filter)',
        'Tracts dropped (Type 1 NaN rate — parks/airports)',
        'Tracts entering overlay (residential only)',
        'Overlay fragments produced',
        'Unique taxi zones with ≥1 fragment',
        'Zones with zero fragments (EWR + island zones)'
    ],
    'Value':   [
        len(tract_gdf),
        int(tract_gdf['zero_vehicle_rate'].isna().sum()),
        len(tract_with_rate),
        len(overlay),
        overlay['LocationID'].nunique(),
        len(set(taxi_gdf['LocationID']) - set(overlay['LocationID']))
    ],
    'Notes':   [
        'After COUNTYFP filter (5411 → 2327)',
        'Total_households = 0 → dropped in Step 6',
        '2228 residential tracts = 2327 − 99 Type 1',
        'Each fragment = one tract-zone intersection polygon',
        '259 = 263 − 4 zero-fragment zones',
        '{1, 103, 104, 105} → filled in Step 19'
    ]
})

(overlay_summary.style
 .set_caption('Table: Spatial Intersection Summary — gpd.overlay() Output')
 .set_properties(**{'text-align': 'left', 'padding': '6px 12px', 'font-size': '13px'})
 .set_table_styles([
     {'selector': 'caption', 'props': [('font-size','14px'),('font-weight','bold'),('padding','8px 0')]},
     {'selector': 'th', 'props': [('background-color','#1D9E75'),('color','white'),('padding','7px 12px')]},
     {'selector': 'tr:nth-child(even)', 'props': [('background-color','#f0f9f4')]},
 ])
 .hide(axis='index'))


### Step 16 — Coverage diagnostic (QA only — does not affect weights)

In [ ]:
# Coverage = fraction of each zone's area covered by residential census tracts
# zone_area (computed in Step 13) is used HERE for QA — it never enters the weighting formula
coverage = (
    overlay.groupby('LocationID')['frag_area'].sum() /
    taxi_gdf.set_index('LocationID')['zone_area']
).reset_index()
coverage.columns = ['LocationID', 'coverage_pct']

zones_no_fragments = set(taxi_gdf['LocationID']) - set(overlay['LocationID'])
low_coverage = coverage[coverage['coverage_pct'] < 0.80]

print(f"Zones with 0 residential fragments (will receive NaN after Step 18): {len(zones_no_fragments)}")
print(f"  → {sorted(zones_no_fragments)}")
print()
print(f"Zones with <80% residential coverage: {len(low_coverage)}")
print()

# Verify low-coverage zones are non-residential using TLC zone names from lookup
# Non-residential zones typically include parks, airports, cemeteries, water areas
NON_RES_KEYWORDS = ['Park', 'Airport', 'Cemetery', 'Bay', 'Island', 'Beach',
                    'Rikers', 'Navy Yard', 'Industrial', 'Track', 'Stadium', 'Meadow']
pattern = '|'.join(NON_RES_KEYWORDS)

low_cov_named = (
    low_coverage
    .merge(taxi_lookup[['LocationID', 'Zone', 'Borough']], on='LocationID', how='left')
)
low_cov_named['likely_non_residential'] = low_cov_named['Zone'].str.contains(
    pattern, case=False, na=False
)

non_res_count = low_cov_named['likely_non_residential'].sum()
res_flagged   = (~low_cov_named['likely_non_residential']).sum()

print(f"Low-coverage zones matching non-residential keywords: {non_res_count}/{len(low_cov_named)}")
if res_flagged > 0:
    print(f"WARNING: {res_flagged} low-coverage zone(s) may be residential — investigate:")
    print(low_cov_named[~low_cov_named['likely_non_residential']][
        ['LocationID', 'Zone', 'Borough', 'coverage_pct']
    ].to_string(index=False))
else:
    print("All low-coverage zones match known non-residential zone types ✓")

print()
# Confirm the exact zero-fragment zones — these are filled in Step 19
print(f"Zones with zero residential fragments (exact): {sorted(zones_no_fragments)}")
print("Their rates will be set to borough mean in Step 19.")


In [ ]:
# Plot 7: 20 lowest residential coverage zones
coverage = (overlay.groupby('LocationID')['frag_area'].sum() /
            taxi_gdf.set_index('LocationID')['zone_area']).reset_index()
coverage.columns = ['LocationID', 'coverage_pct']
low20 = coverage.nsmallest(20, 'coverage_pct').merge(
    taxi_lookup[['LocationID','Zone','Borough']], on='LocationID', how='left'
).sort_values('coverage_pct')

boro_cmap = {**BOROUGH_COLORS, 'EWR': '#888888'}
colors_low = [boro_cmap.get(b, '#888888') for b in low20['Borough']]

fig, ax = plt.subplots(figsize=(9, 7))
bars = ax.barh(range(len(low20)), low20['coverage_pct'] * 100,
               color=colors_low, alpha=0.85, edgecolor='white', linewidth=0.4, height=0.7)
ax.set_yticks(range(len(low20)))
ax.set_yticklabels([f"{r['Zone']} ({r['Borough']})" for _, r in low20.iterrows()], fontsize=8.5)
for bar, val in zip(bars, low20['coverage_pct']):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            f'{val:.1%}', va='center', fontsize=8)
ax.set_xlabel('Residential census tract coverage (%)')
ax.set_title('Figure: 20 Zones with Lowest Residential Coverage\nParks, airports and water areas — structural, not pipeline errors', pad=10)
from matplotlib.patches import Patch
handles = [Patch(facecolor=BOROUGH_COLORS[b], label=b) for b in BOROUGH_ORDER if b in low20['Borough'].values]
ax.legend(handles=handles, loc='lower right', fontsize=8.5, framealpha=0.9)
plt.tight_layout()
plt.show()


### Step 17 — Precision weighted average

In [ ]:
# WHY zone_area is NOT in the weighting formula:
# zone_area is a constant for every fragment within the same LocationID.
# After normalisation (dividing raw_weight by the per-zone sum), the constant cancels exactly.
# Verified on actual shapefiles: max absolute difference = 0.000000000000000 (bit-for-bit identical).
# Only frag_area and precision are needed.

overlay['variance']  = (overlay['moe_zero_vehicle'] / 1.645) ** 2

# Precision = inverse variance: high MOE → low precision → less influence on zone rate
# Fallback to area-only weighting when variance = 0 (perfectly certain ACS estimates)
overlay['precision'] = np.where(
    overlay['variance'] == 0,
    np.nan,
    1 / overlay['variance']
)
overlay['raw_weight'] = np.where(
    overlay['precision'].isna(),
    overlay['frag_area'],
    overlay['frag_area'] * overlay['precision']
)

# Normalise so weights per zone sum to exactly 1
overlay['weight'] = (
    overlay['raw_weight'] /
    overlay.groupby('LocationID')['raw_weight'].transform('sum')
)

overlay.drop(
    columns=['moe_zero_vehicle', 'variance', 'precision', 'raw_weight'],
    inplace=True
)

overlay['weighted_rate'] = overlay['weight'] * overlay['zero_vehicle_rate']

weight_check = overlay.groupby('LocationID')['weight'].sum()

# Weights must sum to exactly 1.0 per zone — if not, the weighted average is wrong
# and all downstream zone rates are incorrect
assert np.allclose(weight_check, 1.0),     f"Weights do not sum to 1.0 — min={weight_check.min():.10f}, max={weight_check.max():.10f}"

print(f"Weight sum check per zone (should all be 1.0):")
print(weight_check.describe().round(6))
print(f"\nMin weight sum: {weight_check.min():.10f}  ✓")
print(f"Max weight sum: {weight_check.max():.10f}  ✓")

In [ ]:
# Plot 8: Area weight vs precision-adjusted weight for a real example zone
# Find zone with most variation between area and precision weights
overlay_temp = overlay.copy()
overlay_temp['area_weight'] = overlay_temp['frag_area'] / overlay_temp.groupby('LocationID')['frag_area'].transform('sum')
overlay_temp['weight_diff'] = abs(overlay_temp['weight'] - overlay_temp['area_weight'])
good_zones = overlay_temp.groupby('LocationID').filter(lambda g: 3 <= len(g) <= 5)
best_zone  = good_zones.groupby('LocationID')['weight_diff'].mean().idxmax()
example    = overlay_temp[overlay_temp['LocationID'] == best_zone].nlargest(5, 'frag_area').reset_index(drop=True)
zone_name  = taxi_lookup[taxi_lookup['LocationID'] == best_zone]['Zone'].values[0]
labels     = example['GEOID'].str[-6:].tolist()
x = np.arange(len(labels))

fig, axes = plt.subplots(1, 2, figsize=(11, 5), sharey=True)
axes[0].bar(x, example['area_weight'] * 100, color='#888780', alpha=0.75, edgecolor='white', width=0.5)
axes[0].set_title('Area-only weighting\n(ignores statistical reliability)', fontsize=10.5)
axes[0].set_ylabel('Weight (%)')
axes[0].set_xticks(x); axes[0].set_xticklabels([f'Tract\n{l}' for l in labels], fontsize=8)
for i, v in enumerate(example['area_weight']):
    axes[0].text(i, v*100 + 0.4, f'{v:.1%}', ha='center', fontsize=9)

axes[1].bar(x, example['weight'] * 100, color='#534AB7', alpha=0.78, edgecolor='white', width=0.5)
axes[1].set_title('Precision-adjusted weighting\n(1/MOE² × area — rewards reliable tracts)', fontsize=10.5)
axes[1].set_xticks(x); axes[1].set_xticklabels([f'Tract\n{l}' for l in labels], fontsize=8)
for i, v in enumerate(example['weight']):
    axes[1].text(i, v*100 + 0.4, f'{v:.1%}', ha='center', fontsize=9)

for ax_ in axes:
    ax_.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}%'))
fig.suptitle(f'Figure: Area vs Precision Weighting — {zone_name} (LocationID {best_zone})\nWeights shift toward statistically reliable tracts when 1/MOE² is applied', fontsize=10.5, y=1.01)
plt.tight_layout()
plt.show()

# np.where fallback note
zero_moe = int((overlay['weight'] == overlay_temp['area_weight']).sum())
print(f"Precision weighting verification:")
print(f"  Weight sums per zone: all = 1.0000000000 ✓")
print(f"  Zero-MOE fallback (area-only) triggered: {zero_moe} fragments")


### Step 18 — Aggregate to one rate per taxi zone

In [ ]:
# Sum weighted contributions per zone → one final zero_vehicle_rate per LocationID
# Weights sum to 1 per zone so groupby sum IS the weighted average
zero_vehicle_rates = (
    overlay
    .groupby('LocationID')['weighted_rate']
    .sum()
    .reset_index()
    .rename(columns={'weighted_rate': 'zero_vehicle_rate'})
)

# 263 shapefile zones − 4 with zero fragments (EWR + 3 islands) = 259 expected
assert len(zero_vehicle_rates) == 259, \
    f"Expected 259 computed zones, got {len(zero_vehicle_rates)} — check overlay for missing zones"

print(f"Zones with computed rate: {len(zero_vehicle_rates)}")
print()
print("Rate statistics across all computed zones:")
print(zero_vehicle_rates['zero_vehicle_rate'].describe().round(4))

In [ ]:
# Aggregated rate statistics
agg_stats = pd.DataFrame({
    'Metric': ['Zones with computed rate', 'Mean zone rate', 'Std dev', 'Min rate', 'Max rate', 'Assertion check'],
    'Value':  [
        f"{len(zero_vehicle_rates)} (expected: 263 − 4 = 259)",
        f"{zero_vehicle_rates['zero_vehicle_rate'].mean():.4f}",
        f"{zero_vehicle_rates['zero_vehicle_rate'].std():.4f}",
        f"{zero_vehicle_rates['zero_vehicle_rate'].min():.4f}",
        f"{zero_vehicle_rates['zero_vehicle_rate'].max():.4f}",
        f"len == 259 ✓"
    ]
})

(agg_stats.style
 .set_caption('Table: Aggregated Zero Vehicle Rate Statistics — 259 Computed Zones')
 .set_properties(**{'text-align': 'left', 'padding': '6px 12px', 'font-size': '13px'})
 .set_table_styles([
     {'selector': 'caption', 'props': [('font-size','14px'),('font-weight','bold'),('padding','8px 0')]},
     {'selector': 'th', 'props': [('background-color','#534AB7'),('color','white'),('padding','7px 12px')]},
     {'selector': 'tr:nth-child(even)', 'props': [('background-color','#f5f4ff')]},
 ])
 .hide(axis='index'))


### Step 19 — Fill NaN zones with borough mean

In [ ]:
# Merge computed rates back onto the taxi zone GeoDataFrame
taxi_gdf = taxi_gdf.merge(zero_vehicle_rates, on='LocationID', how='left')

nan_count = taxi_gdf['zero_vehicle_rate'].isna().sum()
print(f"Zones with NaN before fill: {nan_count}")
print()

nan_zone_detail = (
    taxi_gdf[taxi_gdf['zero_vehicle_rate'].isna()][['LocationID', 'borough']]
    .merge(taxi_lookup[['LocationID', 'Zone']], on='LocationID', how='left')
)
print("Zones receiving borough mean fill:")
print(nan_zone_detail.to_string(index=False))
print()

# Derive IMPUTED_IDS dynamically from actual NaN zones — never hardcoded
# This ensures Phase 5 reflects real data even if a future shapefile version changes
IMPUTED_IDS      = taxi_gdf[taxi_gdf['zero_vehicle_rate'].isna()]['LocationID'].tolist()
IMPUTED_IDS_kept = [z for z in IMPUTED_IDS if z != 1]  # exclude EWR — it is removed in Step 20

print(f"IMPUTED_IDS (all zero-fragment zones, derived):          {sorted(IMPUTED_IDS)}")
print(f"IMPUTED_IDS_kept (island zones that stay in lookup):     {IMPUTED_IDS_kept}")
print()

# Borough mean fill — more accurate than citywide mean (62pp difference across boroughs)
# EWR borough = 'EWR' → not in BOROUGH_MEANS → falls back to citywide mean (fine: excluded in Step 20)
# Zero must never be used as fill: it falsely asserts every household owns a car
for idx, row in taxi_gdf[taxi_gdf['zero_vehicle_rate'].isna()].iterrows():
    fill_value = BOROUGH_MEANS.get(row['borough'], taxi_gdf['zero_vehicle_rate'].mean())
    taxi_gdf.loc[idx, 'zero_vehicle_rate'] = fill_value

assert taxi_gdf['zero_vehicle_rate'].isna().sum() == 0, \
    "NaN values remain after borough mean fill — fill loop failed to cover all zones"

print(f"Fill values used: {BOROUGH_MEANS}")
print(f"Zones with NaN after fill: {taxi_gdf['zero_vehicle_rate'].isna().sum()}  ✓")


In [ ]:
# Plot 9: Borough means — tract level vs zone level comparison
# zero_vehicle_lookup is defined in Step 20 — use taxi_gdf here (NaN-filled rates + borough info)
# taxi_gdf at this point: LocationID, borough, geometry, zone_area, zero_vehicle_rate (all filled)
zone_borough   = taxi_gdf[taxi_gdf['LocationID'] != 1][['LocationID','borough','zero_vehicle_rate']].copy()
zone_borough   = zone_borough.rename(columns={'borough': 'Borough'})
zone_means_val = zone_borough.groupby('Borough')['zero_vehicle_rate'].mean().reindex(BOROUGH_ORDER)

tract_means_val = pd.Series(BOROUGH_MEANS).reindex(BOROUGH_ORDER)
x, w = np.arange(len(BOROUGH_ORDER)), 0.35

fig, ax = plt.subplots(figsize=(9, 5))
b1 = ax.bar(x - w/2, tract_means_val * 100, w,
            label='Tract level (census CSV input)',
            color=[BOROUGH_COLORS[b] for b in BOROUGH_ORDER], alpha=0.50, edgecolor='white', hatch='///')
b2 = ax.bar(x + w/2, zone_means_val * 100, w,
            label='Zone level (spatial join output)',
            color=[BOROUGH_COLORS[b] for b in BOROUGH_ORDER], alpha=0.90, edgecolor='white')

for bars, col in [(b1, tract_means_val), (b2, zone_means_val)]:
    for bar, val in zip(bars, col):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{val:.1%}', ha='center', va='bottom', fontsize=8.5)

ax.set_xticks(x); ax.set_xticklabels(BOROUGH_ORDER)
ax.set_ylabel('Mean zero vehicle rate (%)')
ax.set_title('Figure: Borough Means — Tract Level Input vs Zone Level Output\nOuter boroughs shift down as large low-rate park and suburban zones dominate area weighting', pad=10)
ax.legend(framealpha=0.9); ax.set_ylim(0, 88)
plt.tight_layout()
plt.show()


In [ ]:
# NaN fill summary table — shows exactly which zones were filled and why
nan_fill_zones = taxi_gdf[taxi_gdf['LocationID'].isin(sorted(IMPUTED_IDS))][['LocationID','borough']].merge(
    taxi_lookup[['LocationID','Zone']], on='LocationID', how='left'
)
nan_fill_zones['fill_value'] = nan_fill_zones['borough'].map(
    lambda b: BOROUGH_MEANS.get(b, round(taxi_gdf['zero_vehicle_rate'].mean(), 4))
)
nan_fill_zones['fill_source'] = nan_fill_zones['borough'].map(
    lambda b: f"BOROUGH_MEANS['{b}']" if b in BOROUGH_MEANS else 'citywide mean (EWR — excluded in Step 20)'
)
nan_fill_zones['kept_in_lookup'] = nan_fill_zones['LocationID'].map(lambda z: '✗ Excluded (EWR)' if z == 1 else '✓ Kept')

(nan_fill_zones.style
 .set_caption('Table: NaN Zone Fill Summary — Borough Mean Applied to 4 Zero-Fragment Zones')
 .set_properties(**{'text-align': 'left', 'padding': '6px 12px', 'font-size': '13px'})
 .set_table_styles([
     {'selector': 'caption', 'props': [('font-size','14px'),('font-weight','bold'),('padding','8px 0')]},
     {'selector': 'th', 'props': [('background-color','#BA7517'),('color','white'),('padding','7px 12px')]},
     {'selector': 'tr:nth-child(even)', 'props': [('background-color','#fff8e1')]},
 ])
 .hide(axis='index'))


### Step 20 — Exclude LocationID 1 (Newark Airport EWR)

In [ ]:
# Build the lookup table from the taxi zone GeoDataFrame
zero_vehicle_lookup = taxi_gdf[['LocationID', 'zero_vehicle_rate']].copy()

# Remove Newark Airport — it sits in New Jersey with no NYC census tract coverage
# Any rate assigned to it would be fabricated data entering the model
zero_vehicle_lookup = zero_vehicle_lookup[zero_vehicle_lookup['LocationID'] != 1]

print(f"Zones in lookup: {len(zero_vehicle_lookup)}")
print(f"LocationID 1 still present: {1 in zero_vehicle_lookup['LocationID'].values}")
print()
print("Rate distribution in final lookup:")
print(zero_vehicle_lookup['zero_vehicle_rate'].describe().round(4))

In [ ]:
# EWR exclusion detail table
ewr_table = pd.DataFrame({
    'LocationID':      [1],
    'Zone name':       ['Newark Airport'],
    'Borough':         ['EWR'],
    'State':           ['New Jersey'],
    'In shapefile':    ['✓ Yes'],
    'NYC census tracts cover it': ['✗ No'],
    'Overlay fragments': [0],
    'Rate after fill': [f"citywide mean (fallback — EWR not in BOROUGH_MEANS)"],
    'Action':          ['EXCLUDED — fabricated NJ rate must not enter NYC model']
})

(ewr_table.T.reset_index().style
 .set_caption('Table: Newark Airport (EWR) Exclusion Justification')
 .set_properties(**{'text-align': 'left', 'padding': '6px 12px', 'font-size': '13px'})
 .set_table_styles([
     {'selector': 'caption', 'props': [('font-size','14px'),('font-weight','bold'),('padding','8px 0')]},
     {'selector': 'th', 'props': [('background-color','#D85A30'),('color','white'),('padding','7px 12px')]},
     {'selector': 'tr:nth-child(even)', 'props': [('background-color','#fff3ee')]},
 ])
 .hide(axis='index'))


### Step 21 — Six-assertion output validation

In [ ]:
# Quality gate before zero_vehicle_rate enters the model pipeline
# Any failing assertion points to a specific problem that must be fixed

assert len(zero_vehicle_lookup) == 262, \
    f"Expected 262 zones but got {len(zero_vehicle_lookup)}"

assert zero_vehicle_lookup['zero_vehicle_rate'].isna().sum() == 0, \
    "NaN rates found in lookup table"

assert zero_vehicle_lookup['zero_vehicle_rate'].between(0, 1).all(), \
    "Rate outside [0, 1] range found"

assert zero_vehicle_lookup['LocationID'].nunique() == 262, \
    "Duplicate LocationIDs found in lookup"

assert 1 not in zero_vehicle_lookup['LocationID'].values, \
    "EWR (LocationID 1) still present in lookup"

assert zero_vehicle_lookup['LocationID'].isin(range(2, 264)).all(), \
    "Invalid LocationID found outside range 2 to 263"

print("All 6 assertions passed.")
print()
print("Final lookup table sample:")
print(zero_vehicle_lookup.head(10).to_string(index=False))

In [ ]:
# Six-assertion validation table — the quality gate
validation_table = pd.DataFrame({
    'Assertion': [
        '1. Row count = 262',
        '2. Zero NaN rates',
        '3. All rates in [0, 1]',
        '4. No duplicate LocationIDs',
        '5. LocationID 1 (EWR) absent',
        '6. All LocationIDs in range [2, 263]'
    ],
    'What it checks': [
        '263 shapefile zones − 1 EWR = 262',
        'NaN fill in Step 19 covered all zones',
        'Clip in Step 9 and aggregation are correct',
        'No cartesian product from duplicate lookup rows',
        'EWR exclusion in Step 20 worked',
        'No invalid zone IDs entered from TLC data'
    ],
    'Actual value': [
        str(len(zero_vehicle_lookup)),
        str(zero_vehicle_lookup['zero_vehicle_rate'].isna().sum()),
        str(zero_vehicle_lookup['zero_vehicle_rate'].between(0,1).all()),
        str(zero_vehicle_lookup['LocationID'].nunique()),
        str(1 not in zero_vehicle_lookup['LocationID'].values),
        str(zero_vehicle_lookup['LocationID'].isin(range(2,264)).all())
    ],
    'Expected':  ['262', '0', 'True', '262', 'True', 'True'],
    'Result':    ['✓ PASS', '✓ PASS', '✓ PASS', '✓ PASS', '✓ PASS', '✓ PASS']
})

(validation_table.style
 .set_caption('Table: Six-Assertion Output Validation Gate — All Checks Passed')
 .set_properties(**{'text-align': 'left', 'padding': '6px 12px', 'font-size': '13px'})
 .set_table_styles([
     {'selector': 'caption', 'props': [('font-size','14px'),('font-weight','bold'),('padding','8px 0')]},
     {'selector': 'th', 'props': [('background-color','#2E8B57'),('color','white'),('padding','7px 12px')]},
     {'selector': 'tr:nth-child(even)', 'props': [('background-color','#f0f9f4')]},
     {'selector': 'td:last-child', 'props': [('font-weight','bold'),('color','#085041')]},
 ])
 .hide(axis='index'))


---
## Phase 4: EDA, Checkpoint, Display and Merge

### Step 22 — EDA visualisations

In [ ]:
# Add borough and zone names to the lookup for plotting
plot_df = zero_vehicle_lookup.merge(
    taxi_lookup[['LocationID', 'Borough', 'Zone']],
    on='LocationID',
    how='left'
)

# Plot 1: Histogram — distribution across all 262 zones
fig1 = px.histogram(
    plot_df, x='zero_vehicle_rate', nbins=20, color='Borough',
    title='Distribution of Zero Vehicle Rate Across 262 NYC Taxi Zones',
    labels={'zero_vehicle_rate': 'Zero Vehicle Rate', 'count': 'Number of Zones'}
)
fig1.update_layout(bargap=0.1)
fig1.show()

# Plot 2: Box plot — spread within each borough
fig2 = px.box(
    plot_df, x='Borough', y='zero_vehicle_rate', color='Borough',
    title='Zero Vehicle Rate by Borough',
    labels={'zero_vehicle_rate': 'Zero Vehicle Rate'}
)
fig2.show()

# Plot 3: Top 10 and bottom 10 zones ranked by rate
top10    = plot_df.nlargest(10, 'zero_vehicle_rate')
bottom10 = plot_df.nsmallest(10, 'zero_vehicle_rate')
rank_df  = pd.concat([top10, bottom10]).sort_values('zero_vehicle_rate', ascending=False)

fig3 = px.bar(
    rank_df, x='Zone', y='zero_vehicle_rate', color='Borough',
    title='Top 10 and Bottom 10 Zones by Zero Vehicle Rate',
    labels={'zero_vehicle_rate': 'Zero Vehicle Rate'}
)
fig3.update_layout(xaxis_tickangle=45)
fig3.show()

print("All 3 EDA plots generated.")
print()
print("Expected pattern:")
print("  Top 10 → Manhattan  |  Bottom 10 → Staten Island / outer Queens")
print("  If not: bug in spatial join or NaN fill steps")

In [ ]:
# Plot 11: Final rate distribution with statistical bounds — static paper-ready figure
from scipy.stats import gaussian_kde

zvr_plot = zero_vehicle_lookup['zero_vehicle_rate'].values
q1p, q3p = np.percentile(zvr_plot, [25, 75])
iqr_p = q3p - q1p
mean_p, std_p = zvr_plot.mean(), zvr_plot.std()

fig, ax = plt.subplots(figsize=(9, 5))
ax.hist(zvr_plot, bins=25, color='#534AB7', alpha=0.70, edgecolor='white', linewidth=0.5, density=True)
kde_x = np.linspace(0, 1, 300)
kde = gaussian_kde(zvr_plot)
ax.plot(kde_x, kde(kde_x), color='#1D3A8A', lw=2.2, label='KDE')
ax.axvline(mean_p, color='#D85A30', lw=2.0, label=f'Mean = {mean_p:.1%}')
ax.axvspan(mean_p - std_p, mean_p + std_p, alpha=0.08, color='#D85A30', label=f'±1 std ({std_p:.1%})')
ax.axvline(q1p - 1.5*iqr_p, color='#1D9E75', lw=1.5, ls='--', label=f'IQR fences [{q1p-1.5*iqr_p:.2f}, {q3p+1.5*iqr_p:.2f}]')
ax.axvline(q3p + 1.5*iqr_p, color='#1D9E75', lw=1.5, ls='--')

skew_p = sp_stats.skew(zvr_plot)
ax.text(0.97, 0.95,
    f'n = {len(zvr_plot)} zones\nmean = {mean_p:.1%}\nstd = {std_p:.1%}\nskew = {skew_p:.3f}\noutliers = 0',
    transform=ax.transAxes, va='top', ha='right', fontsize=9.5,
    bbox=dict(boxstyle='round,pad=0.5', facecolor='white', edgecolor='#ddd', alpha=0.9))
ax.set_xlabel('Zero vehicle rate (PULocationID level)')
ax.set_ylabel('Density')
ax.set_title('Figure: Final zero_vehicle_rate Distribution Across 262 NYC Taxi Zones\nGood variance, zero outliers, near-symmetric — ML suitability confirmed', pad=10)
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0%}'))
ax.legend(fontsize=9, framealpha=0.9)
plt.tight_layout()
plt.show()


### Step 23 — Convert to EPSG:4326 for Plotly choropleth map

In [ ]:
# Create a separate display copy in EPSG:4326 (WGS84) for Plotly
# The EPSG:2263 computation copy (taxi_gdf) is never overwritten
taxi_gdf_display = taxi_gdf.to_crs(epsg=4326)
taxi_display_plot = taxi_gdf_display[taxi_gdf_display['LocationID'] != 1].copy()

# Merge zone names and borough for hover display
taxi_display_plot = taxi_display_plot.merge(
    taxi_lookup[['LocationID', 'Zone', 'Borough']],
    on='LocationID', how='left'
)
taxi_display_plot['rate_pct'] = (taxi_display_plot['zero_vehicle_rate'] * 100).round(2)
taxi_display_plot = taxi_display_plot.reset_index(drop=True)
taxi_display_plot['idx'] = taxi_display_plot.index.astype(str)

geojson = json.loads(taxi_display_plot.to_json())
# Set feature IDs to match DataFrame index for Plotly
for feat in geojson['features']:
    feat['id'] = feat['properties']['idx']

fig4 = px.choropleth_mapbox(
    taxi_display_plot,
    geojson=geojson,
    locations='idx',
    color='zero_vehicle_rate',
    color_continuous_scale='RdYlGn_r',
    mapbox_style='carto-positron',
    zoom=10,
    center={'lat': 40.7128, 'lon': -74.0060},
    opacity=0.75,
    title='Zero Vehicle Rate by NYC Taxi Zone (Red = High / Green = Low)',
    hover_data={'Zone': True, 'Borough': True, 'rate_pct': True,
                'LocationID': True, 'zero_vehicle_rate': False, 'idx': False}
)
fig4.update_traces(
    hovertemplate='<b>%{customdata[0]}</b><br>Borough: %{customdata[1]}<br>LocationID: %{customdata[3]}<br>Zero Vehicle Rate: %{customdata[2]:.1f}%<extra></extra>'
)
fig4.update_layout(margin={'r': 0, 't': 40, 'l': 0, 'b': 0})
fig4.show()
print("Hover now shows: Zone name, Borough, LocationID, and Rate %")


### Step 24 — Rename LocationID to PULocationID and save checkpoint

In [ ]:
# The main taxi trip dataset uses PULocationID as the pickup zone identifier
# Renaming here ensures the join key matches exactly in Step 26
# Checkpoint is saved here (post-rename) so the file column name matches the merge key
zero_vehicle_lookup = zero_vehicle_lookup.rename(
    columns={'LocationID': 'PULocationID'}
)

print("Column renamed: LocationID → PULocationID")
print(f"Lookup shape: {zero_vehicle_lookup.shape}")
print(f"Columns: {zero_vehicle_lookup.columns.tolist()}")
print()
print(zero_vehicle_lookup.head(3))

# Save after rename — both file and variable now use PULocationID consistently
zero_vehicle_lookup.to_parquet(OUTPUT_PATH + 'zero_vehicle_rate_by_zone.parquet', index=False)
zero_vehicle_lookup.to_csv(OUTPUT_PATH + 'zero_vehicle_rate_by_zone.csv', index=False)
print(f"\nCheckpoint saved to {OUTPUT_PATH}")
print(f"  zero_vehicle_rate_by_zone.parquet  (PULocationID column — exact float precision)")
print(f"  zero_vehicle_rate_by_zone.csv      (human readable backup)")

In [ ]:
# Checkpoint output summary
checkpoint_summary = pd.DataFrame({
    'File':        ['zero_vehicle_rate_by_zone.parquet', 'zero_vehicle_rate_by_zone.csv'],
    'Format':      ['Parquet (exact float64 precision)', 'CSV (human-readable backup)'],
    'Rows':        [len(zero_vehicle_lookup), len(zero_vehicle_lookup)],
    'Columns':     [str(zero_vehicle_lookup.columns.tolist()), str(zero_vehicle_lookup.columns.tolist())],
    'Key column':  ['PULocationID', 'PULocationID'],
    'Rate range':  [f"[{zero_vehicle_lookup['zero_vehicle_rate'].min():.4f}, {zero_vehicle_lookup['zero_vehicle_rate'].max():.4f}]"] * 2,
    'NaN values':  [0, 0]
})

(checkpoint_summary.style
 .set_caption(f'Table: Checkpoint Files Saved to {OUTPUT_PATH}')
 .set_properties(**{'text-align': 'left', 'padding': '6px 12px', 'font-size': '13px'})
 .set_table_styles([
     {'selector': 'caption', 'props': [('font-size','14px'),('font-weight','bold'),('padding','8px 0')]},
     {'selector': 'th', 'props': [('background-color','#2E8B57'),('color','white'),('padding','7px 12px')]},
 ])
 .hide(axis='index'))


---
## Phase 5: Imputed Zone Evaluation and ML Training Validation

### Part A — Imputed Zone Geographic and Coverage Verification

Three island zones (Governor's Island, Ellis Island, Liberty Island) received the Manhattan borough mean fill in Step 19 — derived dynamically from the overlay output, not hardcoded.  
This section uses actual shapefile geometry and overlay data to verify that the imputation is justified — not just by name, but by measurable geographic and demographic evidence.  
It then decides whether these zones should be kept or removed from the feature lookup.


In [ ]:

# IMPUTED_IDS and IMPUTED_IDS_kept are derived dynamically in Step 19
# They represent zones that received borough mean fill (excluding EWR which is dropped)
# Phase 5 uses these variables directly — no hardcoding needed

# ── A1. Name verification from official TLC lookup ────────────────────────────
imputed_names = taxi_lookup[['LocationID', 'Zone', 'Borough']]
imputed_names = imputed_names[imputed_names['LocationID'].isin(IMPUTED_IDS_kept)].copy()

# Verify zone names contain 'Island' — confirms these are geographically isolated
assert imputed_names['Zone'].str.contains('Island', case=False).all(), \
    f"Expected all imputed zone names to contain 'Island': {imputed_names[['LocationID','Zone']].values.tolist()}"

print("A1 — Imputed zone names (derived from Step 19 dynamically):")
print(imputed_names[['LocationID', 'Zone', 'Borough']].to_string(index=False))
print()
print(f"Zone name check: all contain 'Island' ✓")


In [ ]:
# ── A2. Geographic isolation — distance from imputed zones to nearest mainland zone ──
# taxi_gdf is in EPSG:2263 (NY State Plane, feet) — correct for distance measurement

ISOLATION_THRESHOLD_FT = 100

mainland_union = unary_union(
    taxi_gdf[~taxi_gdf['LocationID'].isin(IMPUTED_IDS_kept + [1])].geometry
)

isolation_results = []
for _, row in taxi_gdf[taxi_gdf['LocationID'].isin(IMPUTED_IDS_kept)].iterrows():
    dist_ft    = row.geometry.distance(mainland_union)
    is_isolated = dist_ft > ISOLATION_THRESHOLD_FT
    assert is_isolated, (
        f"Zone {row['LocationID']} is only {dist_ft:.0f} ft from mainland — "
        f"expected isolated island. Imputation rationale must be reviewed."
    )
    isolation_results.append({
        'LocationID':              row['LocationID'],
        'distance_to_mainland_ft': round(dist_ft, 0),
        'is_isolated':             is_isolated
    })

isolation_df = pd.DataFrame(isolation_results)
print("A2 — Geographic isolation confirmed (distance > 100 ft from mainland):")
print(isolation_df.to_string(index=False))
print("Geographic isolation: PASS ✓")


In [ ]:
# ── A3. Residential census tract coverage — verify NaN is structural, not a bug ──
# overlay holds all fragments from Step 15
# Zero fragments for IMPUTED_IDS_kept confirms the borough mean fill was correct

zones_in_overlay     = set(overlay['LocationID'].unique())
zones_not_in_overlay = [z for z in IMPUTED_IDS_kept if z not in zones_in_overlay]

# Cross-check: dynamically derived IMPUTED_IDS must exactly equal all zero-fragment zones (minus EWR)
all_zero_frag_kept = sorted([z for z in sorted(
    set(taxi_gdf['LocationID']) - zones_in_overlay
) if z != 1])

assert all_zero_frag_kept == sorted(IMPUTED_IDS_kept), (
    f"Zero-fragment zones {all_zero_frag_kept} differ from IMPUTED_IDS_kept {sorted(IMPUTED_IDS_kept)} — "
    f"dynamic derivation in Step 19 does not match overlay result."
)
assert len(zones_not_in_overlay) == len(IMPUTED_IDS_kept), (
    f"Zones {set(IMPUTED_IDS_kept) - set(zones_not_in_overlay)} unexpectedly have "
    f"residential fragments — NaN fill in Step 19 was incorrect."
)

coverage_check = pd.DataFrame({
    'LocationID':            IMPUTED_IDS_kept,
    'residential_fragments': [len(overlay[overlay['LocationID'] == z]) for z in IMPUTED_IDS_kept],
    'got_nan_in_step18':     [z not in zones_in_overlay for z in IMPUTED_IDS_kept],
    'fill_justified':        [z not in zones_in_overlay for z in IMPUTED_IDS_kept]
})

print("A3 — Residential fragment count for imputed zones (must all be 0):")
print(coverage_check.to_string(index=False))
print()
assert (coverage_check['residential_fragments'] == 0).all(), "Imputed zones have unexpected residential fragments"
print("Coverage verification: PASS ✓")


In [ ]:
# ── A4. Fill value plausibility — borough mean must be within Manhattan rate distribution ──
manhattan_rates = (
    zero_vehicle_lookup
    .rename(columns={'PULocationID': 'LocationID'})
    .merge(taxi_lookup[['LocationID', 'Borough']], on='LocationID')
    .query("Borough == 'Manhattan' and LocationID not in @IMPUTED_IDS_kept")
    ['zero_vehicle_rate']
)

fill_value = BOROUGH_MEANS['Manhattan']

assert manhattan_rates.min() <= fill_value <= manhattan_rates.max(), (
    f"Fill value {fill_value:.4f} is outside Manhattan range "
    f"[{manhattan_rates.min():.4f}, {manhattan_rates.max():.4f}]"
)
assert abs(fill_value - manhattan_rates.mean()) <= manhattan_rates.std(), (
    f"Fill value {fill_value:.4f} is more than 1 std from Manhattan mean"
)

plausibility = pd.DataFrame({
    'fill_value':               [round(fill_value, 4)],
    'manhattan_min':            [round(manhattan_rates.min(), 4)],
    'manhattan_mean':           [round(manhattan_rates.mean(), 4)],
    'manhattan_max':            [round(manhattan_rates.max(), 4)],
    'fill_within_range':        [manhattan_rates.min() <= fill_value <= manhattan_rates.max()],
    'fill_within_1std_of_mean': [abs(fill_value - manhattan_rates.mean()) <= manhattan_rates.std()]
})

print("A4 — Fill value plausibility:")
print(plausibility.to_string(index=False))
print("Fill value plausibility: PASS ✓")


In [ ]:
# ── A5. Keep or Remove Decision ─────────────────────────────────────────
# These zones ARE in the official TLC taxi zone shapefile.
# If removed, any df_hourly row with PULocationID in IMPUTED_IDS_kept produces NaN
# after the merge in Step 25, failing the assertion in Step 27.

imputation_rate = len(IMPUTED_IDS_kept) / len(zero_vehicle_lookup)
assert imputation_rate < 0.05, (
    f"Imputation rate {imputation_rate:.1%} exceeds 5% — reconsider strategy"
)

# Zone names derived from lookup — never hardcoded
imputed_zone_names = ', '.join(
    taxi_lookup[taxi_lookup['LocationID'].isin(IMPUTED_IDS_kept)]['Zone'].unique()
)

keep_remove = pd.DataFrame({
    'zone_ids':               [str(IMPUTED_IDS_kept)],
    'zone_names':             [imputed_zone_names],
    'geographic_isolation':   [f"Confirmed — all > {ISOLATION_THRESHOLD_FT} ft from mainland"],
    'residential_population': ['Zero (no census tracts with households)'],
    'fill_value':             [fill_value],
    'fill_plausible':         ['Yes — within Manhattan distribution'],
    'imputation_rate':        [f'{imputation_rate:.1%}'],
    'below_5pct_threshold':   [imputation_rate < 0.05],
    'decision':               ['KEEP — removing causes NaN in df_hourly merge']
})

print("A5 — Keep or Remove Decision:")
print(keep_remove.T.to_string())


### Part B — ML Training Suitability Validation

Each check below runs real code.  
Assertions throw errors if any condition fails — a clean run means every check passes.  
DataFrames are produced for each check so the computed numbers are visible, not just stated.


In [ ]:

zvr   = zero_vehicle_lookup['zero_vehicle_rate']
zvr_a = zvr.values

# ── B1. Completeness and range ───────────────────────────────────────────────
assert zvr.notna().all(),              "NaN values found in zero_vehicle_rate"
assert len(zero_vehicle_lookup) == 262, f"Expected 262 zones, got {len(zero_vehicle_lookup)}"
assert (zvr_a >= 0).all() and (zvr_a <= 1).all(), "Rates outside [0, 1]"

# ── B2. Feature variance — must be large enough to produce useful splits ──────
std = zvr_a.std()
cv  = std / zvr_a.mean()   # coefficient of variation

assert std  > 0.10, f"Std {std:.4f} < 0.10 — near-constant feature, no splitting power"
assert cv   > 0.50, f"CV {cv:.4f} < 0.50 — feature may not discriminate zones sufficiently"

# ── B3. Full distribution summary ────────────────────────────────────────────
pd.DataFrame({
    'count':      [len(zvr_a)],
    'mean':       [zvr_a.mean().round(4)],
    'std':        [std.round(4)],
    'cv':         [cv.round(4)],
    'min':        [zvr_a.min().round(4)],
    'q25':        [np.percentile(zvr_a, 25).round(4)],
    'median':     [np.median(zvr_a).round(4)],
    'q75':        [np.percentile(zvr_a, 75).round(4)],
    'max':        [zvr_a.max().round(4)],
    'range':      [(zvr_a.max() - zvr_a.min()).round(4)],
    'std_ok':     [std > 0.10],
    'cv_ok':      [cv > 0.50]
})


In [ ]:
# ── B4. Skewness and kurtosis ────────────────────────────────────────────────
# Tree-based models (XGBoost, Random Forest) are scale- and skew-invariant.
# Thresholds below are informational. The assertion guards only against extreme cases.
skew = sp_stats.skew(zvr_a)
kurt = sp_stats.kurtosis(zvr_a)   # excess kurtosis (normal = 0)

assert abs(skew) < 2.0, f"Skewness {skew:.4f} exceeds ±2.0 — distribution heavily asymmetric"

pd.DataFrame({
    'skewness':         [round(skew, 4)],
    'excess_kurtosis':  [round(kurt, 4)],
    'interpretation':   ['near-symmetric bimodal (Manhattan vs outer borough clusters)'],
    'tree_model_ok':    [abs(skew) < 2.0],
    'note':             ['XGBoost/RF do not require normally distributed features']
})


In [ ]:
# ── B5. Outlier detection — IQR method ──────────────────────────────────────
q1, q3  = np.percentile(zvr_a, [25, 75])
iqr     = q3 - q1
lo_fence, hi_fence = q1 - 1.5 * iqr, q3 + 1.5 * iqr

outlier_mask = (zvr_a < lo_fence) | (zvr_a > hi_fence)
n_outliers   = outlier_mask.sum()

assert n_outliers == 0, (
    f"{n_outliers} statistical outliers detected outside [{lo_fence:.4f}, {hi_fence:.4f}]. "
    f"Investigate before training."
)

# Zones near the boundary (top and bottom 5 rates) — sanity check
top5    = zero_vehicle_lookup.nlargest(5,  'zero_vehicle_rate')[['PULocationID','zero_vehicle_rate']]
bottom5 = zero_vehicle_lookup.nsmallest(5, 'zero_vehicle_rate')[['PULocationID','zero_vehicle_rate']]

outlier_summary = pd.DataFrame({
    'lower_fence':  [round(lo_fence, 4)],
    'upper_fence':  [round(hi_fence, 4)],
    'n_outliers':   [n_outliers],
    'outliers_ok':  [n_outliers == 0]
})
print("Outlier fences:")
print(outlier_summary.to_string(index=False))
print()
print("Top 5 zones (highest zero vehicle rate):")
print(top5.to_string(index=False))
print()
print("Bottom 5 zones (lowest zero vehicle rate):")
print(bottom5.to_string(index=False))


In [ ]:
# ── B6. Borough-level monotonicity ──────────────────────────────────────────
# Expected ordering: Manhattan > Bronx ≈ Brooklyn > Queens > Staten Island
# Reflects real NYC car ownership geography — violation = pipeline error
bv = (
    zero_vehicle_lookup
    .rename(columns={'PULocationID': 'LocationID'})
    .merge(taxi_lookup[['LocationID', 'Borough']], on='LocationID')
)
bm = bv.groupby('Borough')['zero_vehicle_rate'].mean()

assert bm['Manhattan']     > bm['Bronx'],          "Manhattan ≤ Bronx — spatial join error"
assert bm['Manhattan']     > bm['Brooklyn'],        "Manhattan ≤ Brooklyn — spatial join error"
assert bm['Manhattan']     > bm['Queens'],          "Manhattan ≤ Queens — spatial join error"
assert bm['Manhattan']     > bm['Staten Island'],   "Manhattan ≤ Staten Island — spatial join error"
assert bm['Staten Island'] < bm['Queens'],          "Staten Island ≥ Queens — spatial join error"

# Full borough statistics as a DataFrame
bv.groupby('Borough')['zero_vehicle_rate'].agg(
    zones='count', mean='mean', std='std', min='min', max='max'
).round(4)

In [ ]:
# ── B7. Target leakage check ────────────────────────────────────────────────
# zero_vehicle_rate = ACS B08141_002E / B08141_001E (Census data only)
# No taxi pickup counts were used in its computation → no target leakage
# Verify it is static per zone (one unique value per PULocationID)

rates_per_zone    = zero_vehicle_lookup.groupby('PULocationID')['zero_vehicle_rate'].nunique()
max_rates_per_zone = rates_per_zone.max()

assert max_rates_per_zone == 1, (
    f"Some zones have {max_rates_per_zone} different rate values — "
    f"feature is not static, possible merge error"
)

leakage_check = pd.DataFrame({
    'max_unique_rates_per_zone':  [max_rates_per_zone],
    'all_zones_static':           [(rates_per_zone == 1).all()],
    'feature_source':             ['ACS B08141_002E / B08141_001E — Census data only'],
    'taxi_data_used':             [False],
    'target_leakage_risk':        ['NONE']
})
leakage_check


In [ ]:
# ── B8. Final ML suitability verdict — result column derived from actual checks ──
# Reaching this cell means all assertions above passed.
# Each result is computed from the actual values — not hardcoded strings.

ml_verdict = pd.DataFrame({
    'check':  [
        'Completeness (262 zones, 0 NaN)',
        'Range bounded [0, 1]',
        'Sufficient variance (std > 0.10)',
        'Sufficient discriminability (CV > 0.50)',
        'No statistical outliers (IQR)',
        'Acceptable skewness for tree models (|skew| < 2)',
        'Borough monotonicity (Manhattan > Queens > Staten Island)',
        'Imputed zones within borough distribution',
        'No target leakage (static census feature)',
        'Imputation rate below 5% threshold'
    ],
    'result': [
        'PASS' if zvr.notna().all() and len(zero_vehicle_lookup) == 262         else 'FAIL',
        'PASS' if (zvr_a >= 0).all() and (zvr_a <= 1).all()                     else 'FAIL',
        'PASS' if zvr_a.std() > 0.10                                             else 'FAIL',
        'PASS' if (zvr_a.std() / zvr_a.mean()) > 0.50                           else 'FAIL',
        'PASS' if n_outliers == 0                                                else 'FAIL',
        'PASS' if abs(skew) < 2.0                                                else 'FAIL',
        'PASS' if bm['Manhattan'] > bm['Queens'] > bm['Staten Island']           else 'FAIL',
        'PASS' if manhattan_rates.min() <= fill_value <= manhattan_rates.max()   else 'FAIL',
        'PASS' if max_rates_per_zone == 1                                        else 'FAIL',
        'PASS' if imputation_rate < 0.05                                         else 'FAIL',
    ],
    'note': [
        f'{len(zero_vehicle_lookup)}/262 zones, {int(zvr.isna().sum())} NaN',
        f'min={zvr_a.min():.4f}, max={zvr_a.max():.4f}',
        f'std={zvr_a.std():.4f}',
        f'cv={zvr_a.std()/zvr_a.mean():.4f}',
        f'{n_outliers} outliers found',
        f'skew={round(skew, 4)}',
        f'Manhattan={bm["Manhattan"]:.4f} > Queens={bm["Queens"]:.4f} > SI={bm["Staten Island"]:.4f}',
        f'fill={fill_value:.4f} within [{manhattan_rates.min():.4f}, {manhattan_rates.max():.4f}]',
        'Derived from ACS B08141 only — no taxi data used',
        f'{imputation_rate:.1%} ({len(IMPUTED_IDS_kept)}/{len(zero_vehicle_lookup)} zones)',
    ]
})

all_passed = (ml_verdict['result'] == 'PASS').all()
assert all_passed, f"Failed checks: {ml_verdict[ml_verdict['result']=='FAIL']['check'].tolist()}"

print(f"All {len(ml_verdict)} ML suitability checks: {'PASS' if all_passed else 'FAIL'}")
print()
ml_verdict


### Step 25 — Merge zero_vehicle_rate into df_hourly

In [ ]:
# Load the main zone-hour dataset from TLC preprocessing
df_hourly = pd.read_parquet(HOURLY_PATH)
print(f"df_hourly shape before filtering: {df_hourly.shape}")
print(f"Columns before: {df_hourly.columns.tolist()}")
print()

# Remove EWR (LocationID 1) — sits in New Jersey with no NYC census tract coverage
df_hourly = df_hourly[df_hourly['PULocationID'] != 1].copy()
print(f"df_hourly shape after removing EWR: {df_hourly.shape}")
print()

# Left merge preserves all existing rows in df_hourly
df_hourly = df_hourly.merge(zero_vehicle_lookup, on='PULocationID', how='left')
print(f"df_hourly shape after merge:  {df_hourly.shape}")
print(f"New column added: zero_vehicle_rate")
print()

# Drop raw datetime — temporal features already extracted (hour, day_of_week, month)
# tpep_pickup_datetime is not a model feature and must not enter training
df_hourly.drop(columns=['tpep_pickup_datetime'], inplace=True, errors='ignore')

assert 'tpep_pickup_datetime' not in df_hourly.columns,     "tpep_pickup_datetime was not dropped — check column names in HOURLY_PATH"

print(f"Dropped tpep_pickup_datetime — not a model feature  ✓")
print(f"Columns after drop: {df_hourly.columns.tolist()}")
print()
print("Sample of merged data:")
print(df_hourly[['PULocationID', 'date', 'hour', 'pickup_count', 'zero_vehicle_rate']].head(5))

In [ ]:
# Merge summary table
merge_summary = pd.DataFrame({
    'Stage':    ['df_hourly before EWR filter', 'df_hourly after EWR filter', 'df_hourly after merge', 'datetime column dropped'],
    'Shape':    ['Loaded from HOURLY_PATH', 'EWR (LocationID 1) rows removed', f'{df_hourly.shape}', 'tpep_pickup_datetime removed'],
    'Action':   ['pd.read_parquet(HOURLY_PATH)', "df_hourly[df_hourly['PULocationID'] != 1]",
                 "df_hourly.merge(zero_vehicle_lookup, on='PULocationID', how='left')",
                 "drop(columns=['tpep_pickup_datetime'], errors='ignore')"],
    'New columns': ['—', '—', 'zero_vehicle_rate', '—']
})

(merge_summary.style
 .set_caption('Table: Step 25 — Merge Operations Summary')
 .set_properties(**{'text-align': 'left', 'padding': '6px 12px', 'font-size': '13px'})
 .set_table_styles([
     {'selector': 'caption', 'props': [('font-size','14px'),('font-weight','bold'),('padding','8px 0')]},
     {'selector': 'th', 'props': [('background-color','#534AB7'),('color','white'),('padding','7px 12px')]},
     {'selector': 'tr:nth-child(even)', 'props': [('background-color','#f5f4ff')]},
 ])
 .hide(axis='index'))


In [ ]:
# Diagnose: which PULocationIDs in df_hourly don't match the lookup?
missing_zones = df_hourly[df_hourly['zero_vehicle_rate'].isna()]['PULocationID'].unique()
missing_counts = df_hourly[df_hourly['zero_vehicle_rate'].isna()].groupby('PULocationID').size()

print(f"Unique zones with NaN rate: {sorted(missing_zones)}")
print(f"\nRow counts per missing zone:")
print(missing_counts.sort_index())
print(f"\nTotal NaN rows: {df_hourly['zero_vehicle_rate'].isna().sum()}")
print(f"Percentage of data: {100 * df_hourly['zero_vehicle_rate'].isna().sum() / len(df_hourly):.1f}%")

In [ ]:
# Diagnostic results table
nan_count_diag = df_hourly['zero_vehicle_rate'].isna().sum()
diag_table = pd.DataFrame({
    'Check':  ['NaN zones after merge', 'NaN rows (% of data)', 'Root cause'],
    'Result': [
        f"{len(df_hourly[df_hourly['zero_vehicle_rate'].isna()]['PULocationID'].unique())} zones",
        f"{nan_count_diag} rows ({100*nan_count_diag/len(df_hourly):.2f}%)" if len(df_hourly) > 0 else 'N/A',
        'None — all PULocationIDs matched the lookup ✓' if nan_count_diag == 0 else 'Unmatched PULocationIDs — investigate'
    ],
    'Expected': ['0 zones', '0 rows (0.00%)', 'Clean merge ✓'],
    'Status': ['✓ PASS' if nan_count_diag == 0 else '✗ FAIL'] * 3
})

(diag_table.style
 .set_caption('Table: Post-Merge Diagnostic — NaN Check on zero_vehicle_rate')
 .set_properties(**{'text-align': 'left', 'padding': '6px 12px', 'font-size': '13px'})
 .set_table_styles([
     {'selector': 'caption', 'props': [('font-size','14px'),('font-weight','bold'),('padding','8px 0')]},
     {'selector': 'th', 'props': [('background-color','#2E8B57'),('color','white'),('padding','7px 12px')]},
 ])
 .hide(axis='index'))


### Step 26 — Post-merge consistency check

In [ ]:
# zero_vehicle_rate is a static zone property — same value for every row in the same zone
# If any zone has more than one unique rate value the merge created duplicate rows silently
rate_per_zone = df_hourly.groupby('PULocationID')['zero_vehicle_rate'].nunique()
inconsistent  = (rate_per_zone > 1).sum()

assert inconsistent == 0, f"{inconsistent} zones have inconsistent rates — check lookup for duplicates"

print(f"Zones with inconsistent rates: {inconsistent}")
print("Consistency check passed — every zone has exactly one unique rate value.")

In [ ]:
# Consistency check results table
rate_per_zone    = df_hourly.groupby('PULocationID')['zero_vehicle_rate'].nunique()
inconsistent_count = int((rate_per_zone > 1).sum())
consistency_table = pd.DataFrame({
    'Check':    ['Zones checked', 'Max unique rates per zone', 'Zones with > 1 rate', 'Consistency verdict'],
    'Value':    [len(rate_per_zone), int(rate_per_zone.max()), inconsistent_count,
                 'PASS — feature is static ✓' if inconsistent_count == 0 else 'FAIL — merge error'],
    'Expected': [262, 1, 0, 'PASS ✓'],
    'Result':   ['✓', '✓', '✓', '✓ PASS'] if inconsistent_count == 0 else ['✓', '✗', '✗', '✗ FAIL']
})

(consistency_table.style
 .set_caption('Table: Post-Merge Consistency Check — zero_vehicle_rate Static per Zone')
 .set_properties(**{'text-align': 'left', 'padding': '6px 12px', 'font-size': '13px'})
 .set_table_styles([
     {'selector': 'caption', 'props': [('font-size','14px'),('font-weight','bold'),('padding','8px 0')]},
     {'selector': 'th', 'props': [('background-color','#2E8B57'),('color','white'),('padding','7px 12px')]},
     {'selector': 'tr:nth-child(even)', 'props': [('background-color','#f0f9f4')]},
 ])
 .hide(axis='index'))


### Step 27 — Post-merge null check and feature scaling decision

In [ ]:
# Any NaN in zero_vehicle_rate means a PULocationID in df_hourly was not found in the lookup
# Most likely cause: EWR trips not fully removed during TLC preprocessing
null_count = df_hourly['zero_vehicle_rate'].isna().sum()
assert null_count == 0, f"{null_count} NaN values in zero_vehicle_rate — check for unexpected PULocationIDs"

assert df_hourly['zero_vehicle_rate'].between(0, 1).all(), "Rate out of [0, 1] in df_hourly"

print(f"NaN values in zero_vehicle_rate: {null_count}")
print(f"All rates in [0, 1]:             {df_hourly['zero_vehicle_rate'].between(0, 1).all()}")
print()
print("Feature scaling decision:")
print("  zero_vehicle_rate is already bounded [0, 1] by mathematical design.")
print("  No StandardScaler is needed for tree-based models (XGBoost, Random Forest).")
print("  Tree models find their own split thresholds — scale-invariant.")
print("  If a linear baseline model is added later: apply StandardScaler before that model only.")
print()
print("Temporal limitation (document in Chapter 3 Limitations):")
print("  Census ACS data covers the 2020–2024 reference period.")
print("  Taxi trip data covers January–March 2025.")
print("  NYC car ownership patterns are structurally stable year-on-year.")
print("  Mismatch severity: LOW.")
print()

# Save enriched df_hourly — preserves merge result in case of kernel restart
# Next pipeline steps (weather, subway) should load this file as their input
df_hourly.to_parquet(OUTPUT_PATH + 'taxi_demand_with_census.parquet', index=False)
print(f"Enriched df_hourly saved to: {OUTPUT_PATH}taxi_demand_with_census.parquet")
print(f"Final shape: {df_hourly.shape}")
print(f"Final columns: {df_hourly.columns.tolist()}")

In [ ]:
# Final pipeline output summary
final_summary = pd.DataFrame({
    'Property':     [
        'Feature name',
        'Output file',
        'Zones in lookup',
        'Rate range',
        'NaN values',
        'Feature source',
        'Scaling for tree models',
        'Scaling for linear models',
        'Temporal note',
        'Next pipeline step'
    ],
    'Value': [
        'zero_vehicle_rate',
        f'{OUTPUT_PATH}taxi_demand_with_census.parquet',
        f'{len(zero_vehicle_lookup)} zones (PULocationID 2–263)',
        f"[{zero_vehicle_lookup['zero_vehicle_rate'].min():.4f}, {zero_vehicle_lookup['zero_vehicle_rate'].max():.4f}]",
        f"{df_hourly['zero_vehicle_rate'].isna().sum()} ✓",
        'Census ACS B08141 (2020–2024) — zero taxi data used',
        'Not needed — feature is already bounded [0, 1]',
        'Apply StandardScaler before linear baseline models only',
        'ACS 2020–2024 vs trips Jan–Mar 2025 — low mismatch severity',
        'MTA subway station density preprocessing'
    ]
})

(final_summary.style
 .set_caption('Table: Final Pipeline Output Summary — zero_vehicle_rate Ready for Model Training')
 .set_properties(**{'text-align': 'left', 'padding': '6px 12px', 'font-size': '13px'})
 .set_table_styles([
     {'selector': 'caption', 'props': [('font-size','14px'),('font-weight','bold'),('padding','8px 0')]},
     {'selector': 'th', 'props': [('background-color','#534AB7'),('color','white'),('padding','7px 12px')]},
     {'selector': 'tr:nth-child(even)', 'props': [('background-color','#f5f4ff')]},
     {'selector': 'tr:first-child', 'props': [('font-weight','bold'),('background-color','#e8e8ff')]},
 ])
 .hide(axis='index'))
